In [1]:
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import date, datetime, timedelta
import concurrent.futures
import base64
import os
import time
import duckdb
con = duckdb.connect("pbp.duckdb")
# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [279]:



#only run the first time ever
# con.execute(r"""
#     CREATE TABLE game_plays AS
#     SELECT * FROM read_csv_auto(
#         'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Base Data\\Play By Play\\history_play_by_play.csv'
#     );
# """)



## Game Summary Data


In [280]:
history_summary = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Base Data\Game Summary\all_game_summary.csv")
history_summary = history_summary[['game_id', 'season_id', 'awayTeam.id', 'awayTeam.abbrev', 'homeTeam.id', 'homeTeam.abbrev', 'awayTeam.score',
                                   'homeTeam.score', 'gameState', 'link', 'game_date_time', 'game_date']]
history_summary.head()


,game_id,season_id,awayTeam.id,awayTeam.abbrev,homeTeam.id,homeTeam.abbrev,awayTeam.score,homeTeam.score,gameState,link,game_date_time,game_date
0,2013020001,20132014,10,TOR,8,MTL,4,3,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 19:00:00-04:00,10/1/2013
1,2013020002,20132014,15,WSH,16,CHI,4,6,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 20:00:00-04:00,10/1/2013
2,2013020003,20132014,52,WPG,22,EDM,5,4,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 22:00:00-04:00,10/1/2013
3,2013020004,20132014,10,TOR,4,PHI,3,1,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-02 19:30:00-04:00,10/2/2013
4,2013020005,20132014,7,BUF,17,DET,1,2,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-02 20:00:00-04:00,10/2/2013


## Standings Data

In [281]:

final_standings = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\history_final_standings.csv")
final_standings.tail()



,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points
305,20242025,BOS,Boston Bruins,Eastern,15,82,76,0.402439,0.463415,33,-50,222,272,41,20,41,13,-4,OT,1,0.487805,0.317073,2.707317,3.317073,76.0
306,20242025,PHI,Philadelphia Flyers,Eastern,16,82,76,0.402439,0.463415,27,-48,238,286,41,20,41,13,-1,L,2,0.487805,0.317073,2.902439,3.487805,76.0
307,20242025,NSH,Nashville Predators,Western,14,82,68,0.365854,0.414634,28,-60,214,274,41,20,41,10,-12,W,1,0.487805,0.243902,2.609756,3.341463,68.0
308,20242025,CHI,Chicago Blackhawks,Western,15,82,61,0.304878,0.371951,23,-70,226,296,41,15,41,10,-7,W,2,0.365854,0.243902,2.756098,3.609756,61.0
309,20242025,SJS,San Jose Sharks,Western,16,82,52,0.243902,0.317073,18,-105,210,315,41,12,41,8,-24,L,1,0.292683,0.195122,2.560976,3.841463,52.0


## Play by Play Data

In [282]:


all_pbp = con.execute("SELECT * FROM game_plays").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [283]:
all_pbp.tail()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,period_number,period_type,max_regulation_periods,event_team_id,details.losingPlayerId,winning_player_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,shooting_player,goalie,away_sog,home_sog,player_id,reason,blocking_player,type_code,desc_key,duration,committed_by,drawn_by,secondary_reason,pptReplayUrl,scoring_player,scoring_player_total,assist_1,assist1_total,away_score,home_score,details.highlightClipSharingUrl,details.highlightClip,details.discreteClip,assist_2,assist2_total,served_by,details.discreteClipFr,details.highlightClipSharingUrlFr,details.highlightClipFr,situation,goalie_situation,game_in_seconds
4322685,20242025,2024021312,1096,18:56:00,01:04:00,1551.0,left,504,giveaway,735,3,REG,3,29.0,NaN,NaN,48.0,-34.0,O,NaN,NaN,None,NaN,NaN,NaN,NaN,8482660.0,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,1136
4322686,20242025,2024021312,1086,19:20:00,00:40:00,1551.0,left,506,shot-on-goal,736,3,REG,3,29.0,NaN,NaN,37.0,31.0,O,NaN,NaN,slap,8483485.0,8477405.0,36.0,20.0,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,1160
4322687,20242025,2024021312,87,19:27:00,00:33:00,1551.0,left,507,missed-shot,740,3,REG,3,2.0,NaN,NaN,-49.0,-22.0,O,NaN,NaN,slap,8474709.0,8482982.0,NaN,NaN,NaN,wide-left,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,1167
4322688,20242025,2024021312,408,20:00:00,00:00:00,1551.0,left,521,period-end,746,3,REG,3,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,1200
4322689,20242025,2024021312,412,20:00:00,00:00:00,1551.0,left,524,game-end,750,3,REG,3,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,1200


In [284]:
# 1. Build pandas DataFrame
home_away = history_summary[['season_id','game_id', 'homeTeam.id', 'awayTeam.id']]
home_away = home_away.rename(columns={
    'homeTeam.id': 'home_team', 
    'awayTeam.id': 'away_team'
})

# 2. Store permanently in DuckDB
con.execute("DROP TABLE IF EXISTS home_away;")
con.execute("CREATE TABLE home_away AS SELECT * FROM home_away;")

#3. Join with game_plays (the correct DuckDB table name is home_away)
game_plays = con.execute("""
    SELECT gp.season_id, 
                    gp.game_id, 
                    gp.eventId, 
                    gp.timeInPeriod, 
                    gp.timeRemaining,
                    gp.typeCode, 
                    gp.typeDescKey,
                    gp.sortOrder, 
                    gp.period_number, 
                    gp.event_team_id as team_id,
                    gp.x_coord,
                    gp.y_coord, 
                    gp.zone_code, 
                    gp.hitter, 
                    gp.hittee, 
                    gp.shot_type,
                    gp.goalie, 
                    gp.away_sog, 
                    gp.home_sog, 
                    gp.player_id,
                    gp.reason,  
                    gp.blocking_player, 
                    gp.scoring_player,
                    gp.situation, 
                    gp.goalie_situation, 
                    gp.game_in_seconds, 
                    ha.home_team,
                    ha.away_team,
                    ha.season_id,
                    ha.game_id
    FROM game_plays gp
    LEFT JOIN home_away ha
        ON gp.game_id = ha.game_id
       AND gp.season_id = ha.season_id
""").df()

game_plays.tail()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
4742758,20242025,2024021038,526,19:37:00,00:23:00,525,takeaway,289,1,24.0,-93.0,-7.0,D,NaN,NaN,None,NaN,NaN,NaN,8484153.0,None,NaN,NaN,5 on 5,in net,1177,59,24,20242025,2024021038
4742759,20242025,2024021038,519,19:42:00,00:18:00,508,blocked-shot,290,1,24.0,-64.0,-12.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8483431.0,NaN,5 on 5,in net,1182,59,24,20242025,2024021038
4742760,20242025,2024021038,202,19:43:00,00:17:00,507,missed-shot,291,1,24.0,-92.0,-18.0,O,NaN,NaN,wrist,8478872.0,NaN,NaN,NaN,failed-bank-attempt,NaN,NaN,5 on 5,in net,1183,59,24,20242025,2024021038
4742761,20242025,2024021038,26,19:44:00,00:16:00,516,stoppage,292,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,goalie-stopped-after-sog,NaN,NaN,5 on 5,in net,1184,59,24,20242025,2024021038
4742762,20242025,2024021038,73,19:44:00,00:16:00,502,faceoff,294,1,59.0,-69.0,-22.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,1184,59,24,20242025,2024021038


In [285]:

dfs = [history_summary, game_plays]

exclude_cols = ["x_coord", "y_coord"]

for i, df in enumerate(dfs):
    float_cols = df.select_dtypes(include='float').columns
    float_cols = float_cols.difference(exclude_cols)
    dfs[i][float_cols] = df[float_cols].astype("Int64")




## Teams

In [286]:
teams = history_summary[['homeTeam.id', 'homeTeam.abbrev']]
teams=teams.drop_duplicates()
teams = teams.rename(columns={'homeTeam.id':'team_id', 'homeTeam.abbrev':'tri_code'})
teams.head()

,team_id,tri_code
0,8,MTL
1,16,CHI
2,22,EDM
3,4,PHI
4,17,DET


## Goalies Data

In [287]:
# #Goalie Stats
# goalies_individual_games = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\all_goalie_games.csv")
# team_goalie_agg_stats = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\history_goalie_agg_stats.csv")
# team_goalie_agg_stats = team_goalie_agg_stats.rename(columns={'season':'season_id'})

# team_goalie_agg_stats.head()

con.execute(r"""
    CREATE OR REPLACE TABLE all_goalie_games AS
    SELECT
        season       AS season_id,
        gameId      AS game_id,
        *
    EXCLUDE (season, gameId)
    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Goalies\\all_goalie_games.csv'
    )
""")
goalies_individual_games = con.execute("""
    SELECT *
    FROM all_goalie_games
""").df()

goalies_individual_games.tail()

,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
32172,20252026,2025020117,1,1,Joel Hofer,8480981,STL,477,0,0,0,22.64150,3,0,0.50000,6,3
32173,20252026,2025020275,1,1,Alex Lyon,8479312,BUF,370,0,0,0,19.45945,1,0,0.33333,3,2
32174,20252026,2025020341,1,1,Logan Thompson,8480313,WSH,1099,0,1,0,13.10282,2,0,0.33333,6,4
32175,20252026,2025020362,1,1,Dustin Wolf,8481692,CGY,352,0,1,0,30.68181,1,0,0.25000,4,3
32176,20252026,2025020259,1,0,Pyotr Kochetkov,8481611,CAR,216,0,0,0,0.00000,0,0,NaN,0,0


## Special Teams Data

In [288]:
# all_pk = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pk_by_game_all.csv")
# all_pp = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pp_by_game_all.csv")
# all_pp = all_pp.rename(columns={'season':'season_id'})
# all_pk = all_pk.rename(columns={'season':'season_id'})


con.execute(r"""
    CREATE OR REPLACE TABLE all_pp AS
    SELECT
        season       AS season_id,
        gameId       AS game_id,
        teamId       AS team_id,
        *
    EXCLUDE (season, gameId, teamId)
    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Team Stats\\pp_by_game_all.csv'
    )
    ORDER BY game_id;
""")



con.execute(r"""
    CREATE OR REPLACE TABLE all_pk AS
    SELECT
        season       AS season_id,
        gameId       AS game_id,
        teamId       AS team_id,
        *
    EXCLUDE (season, gameId,teamId)
    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Team Stats\\pk_by_game_all.csv'
    )
    ORDER BY game_id;
""")

all_pp = con.execute("""
    SELECT *
    FROM all_pp
""").df()

all_pk = con.execute("""
    SELECT *
    FROM all_pk
""").df()



## Clinch Data

In [289]:
clinch_history_full = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\full_clinch_history.csv")




clinch_history_full.head()

,last_ran_date,clinch_indicator,tri_code,clinch_type,clinch_date,today,season_id
0,10/1/2013,True,BUF,e,3/16/2014,4/13/2014,20132014
1,10/1/2013,True,BOS,x,3/21/2014,4/13/2014,20132014
2,10/1/2013,True,STL,x,3/22/2014,4/13/2014,20132014
3,10/1/2013,True,EDM,e,3/24/2014,4/13/2014,20132014
4,10/1/2013,True,SJS,x,3/24/2014,4/13/2014,20132014


In [290]:
# # clinch_history = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\clinch_history_data.csv")
# # clinch_25 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\clinch_history.csv")
# clinch_history_full = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\full_clinch_history.csv")
# clinch_history_full=clinch_history_full.rename(columns={'season':'season_id'})
# # clinch_history_full['clinch_date'] = pd.to_datetime(
# #     clinch_history_full['clinch_date'], errors='coerce'
# # ).dt.strftime('%Y-%m-%d')
# clinch_history_full.tail()




con.execute(r"""
    CREATE OR REPLACE TABLE full_clinch_history  AS
    SELECT


        *

    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Base Data\\Standings\\full_clinch_history.csv'
    )
    ORDER BY season_id;
""")

clinch_history = con.execute("""
    SELECT *
    FROM full_clinch_history
""").df()

clinch_history=clinch_history[['tri_code', 'clinch_type', 'clinch_date', 'season_id']]
clinch_history.head()





,tri_code,clinch_type,clinch_date,season_id
0,BUF,e,2014-03-16,20132014
1,BOS,x,2014-03-21,20132014
2,STL,x,2014-03-22,20132014
3,EDM,e,2014-03-24,20132014
4,SJS,x,2014-03-24,20132014


In [291]:
clinch_history=clinch_history.merge(teams, on='tri_code', how='left')

clinch_history.head()


,tri_code,clinch_type,clinch_date,season_id,team_id
0,BUF,e,2014-03-16,20132014,7
1,BOS,x,2014-03-21,20132014,6
2,STL,x,2014-03-22,20132014,19
3,EDM,e,2014-03-24,20132014,22
4,SJS,x,2014-03-24,20132014,28


### Goalie Stats

In [292]:
all_goalie_games = goalies_individual_games.copy()

# ----- Weighted metric helpers -----

def weighted_save_pct(group):
    shots = group['shotsAgainst'].sum()
    if shots == 0 or pd.isna(shots):
        return np.nan
    return (group['savePct'] * group['shotsAgainst']).sum() / shots

def weighted_gaa(group):
    toi = group['timeOnIce'].sum()
    if toi == 0 or pd.isna(toi):
        return np.nan
    return (group['goalsAgainstAverage'] * group['timeOnIce']).sum() / toi


# ----- Build final individual goalie metrics -----

goalies_individual = (
    all_goalie_games
    .groupby(['goalieFullName', 'season_id', 'teamAbbrev'], dropna=False)
    .apply(lambda g: pd.Series({
        'i_savePct': weighted_save_pct(g),
        'i_GAA': weighted_gaa(g),

        # Totals
        'i_gamesPlayed': g['gamesPlayed'].sum(),
        'i_gamesStarted': g['gamesStarted'].sum(),
        'i_wins': g['wins'].sum(),
        'i_losses': g['losses'].sum(),
        'i_otlosses': g['otLosses'].sum(),
        'i_shutouts': g['shutouts'].sum(),
        'i_saves': g['saves'].sum(),
        'i_shotsAgainst': g['shotsAgainst'].sum(),
        'i_timeOnIce': g['timeOnIce'].sum(),
    }))
    .reset_index()
    .sort_values("teamAbbrev")
)

goalies_individual.head()

C:\Users\kjchr\AppData\Local\Temp\ipykernel_26016\2209191367.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,goalieFullName,season_id,teamAbbrev,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst,i_timeOnIce
882,Lukas Dostal,20222023,ANA,0.901320,3.782754,19.0,17.0,4.0,10.0,3.0,0.0,612.0,679.0,63763.0
480,Frederik Andersen,20142015,ANA,0.914342,2.376260,54.0,53.0,35.0,12.0,5.0,3.0,1313.0,1436.0,186343.0
479,Frederik Andersen,20132014,ANA,0.923369,2.293819,28.0,24.0,20.0,5.0,0.0,0.0,723.0,783.0,94166.0
1048,Olle Eriksson Ek,20222023,ANA,0.871790,4.688720,1.0,1.0,0.0,0.0,1.0,0.0,34.0,39.0,3839.0
428,Dustin Tokarski,20162017,ANA,1.000000,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,587.0


In [293]:
#last 10 games
last_10 = all_goalie_games.sort_values(['goalieFullName','game_id'], ascending=[True, False])
last_10 = last_10.groupby("goalieFullName").head(10)

last_10.head()


,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
24621,20222023,2022020863,1,1,Aaron Dell,8477180,SJS,3567,0,1,0,2.01850,25,0,0.92592,27,2
24396,20222023,2022020847,1,0,Aaron Dell,8477180,SJS,1376,0,0,0,0.00000,8,0,1.00000,8,0
25224,20222023,2022020395,1,1,Aaron Dell,8477180,SJS,3537,0,1,0,5.08905,32,0,0.86486,37,5
24549,20222023,2022020363,1,1,Aaron Dell,8477180,SJS,3477,0,1,0,2.07075,29,0,0.93548,31,2
22184,20212022,2021020744,1,1,Aaron Dell,8477180,BUF,3587,0,1,0,5.01812,42,0,0.89361,47,5


In [294]:
last_10 = last_10.groupby(['goalieFullName', 'season_id','teamAbbrev']).agg(
    i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * all_goalie_games.loc[x.index, 'shotsAgainst']).sum() / all_goalie_games.loc[x.index, 'shotsAgainst'].sum()),
    i_GAA=pd.NamedAgg(column='goalsAgainstAverage', aggfunc=lambda x: (x * all_goalie_games.loc[x.index, 'timeOnIce']).sum() / all_goalie_games.loc[x.index, 'timeOnIce'].sum()),
    i_gamesPlayed=('gamesPlayed', 'sum'),
    i_gamesStarted=('gamesStarted', 'sum'),
    i_wins=('wins', 'sum'),
    i_losses = ('losses', 'sum'),
    i_otlosses = ('otLosses', 'sum'),
    i_shutouts=('shutouts', 'sum'),
    i_saves=('saves', 'sum'),
    i_shotsAgainst=('shotsAgainst', 'sum'),
).sort_values("teamAbbrev").reset_index()
last_10.head()

C:\Users\kjchr\AppData\Local\Temp\ipykernel_26016\2120747135.py:2: RuntimeWarning: invalid value encountered in scalar divide
  i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * all_goalie_games.loc[x.index, 'shotsAgainst']).sum() / all_goalie_games.loc[x.index, 'shotsAgainst'].sum()),


,goalieFullName,season_id,teamAbbrev,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst
0,Ilya Bryzgalov,20142015,ANA,0.846664,4.188372,8,6,1,4,1,0,127,150
1,Lukas Dostal,20252026,ANA,0.903444,2.793246,10,10,8,2,0,0,262,290
2,Olle Eriksson Ek,20222023,ANA,0.871790,4.688720,1,1,0,0,1,0,34,39
3,Petr Mrazek,20252026,ANA,0.881078,3.689035,6,6,3,3,0,0,163,185
4,Reto Berra,20172018,ANA,0.926315,2.308321,5,1,1,1,0,0,88,95


In [295]:
goalie_games_stats = all_goalie_games.groupby(['teamAbbrev', 'season_id']).agg(
    team_gamesPlayed=('gamesPlayed', 'sum'),
    team_gamesStarted=('gamesStarted', 'sum'),
    team_wins=('wins', 'sum'),
    team_losses=('losses', 'sum'),
    team_otlosses=('otLosses', 'sum'),
    team_shutouts=('shutouts', 'sum'),
    team_saves=('saves', 'sum'),
    team_shotsAgainst=('shotsAgainst', 'sum'),
    team_timeOnIce=("timeOnIce", 'sum'),
).reset_index()

goalie_games_stats.head()

,teamAbbrev,season_id,team_gamesPlayed,team_gamesStarted,team_wins,team_losses,team_otlosses,team_shutouts,team_saves,team_shotsAgainst,team_timeOnIce
0,ANA,20132014,86,82,54,20,8,6,2150,2349,297848
1,ANA,20142015,90,82,51,24,7,4,2146,2359,298955
2,ANA,20152016,92,82,46,25,11,8,2070,2253,296936
3,ANA,20162017,92,82,46,23,13,8,2232,2424,297199
4,ANA,20172018,93,82,44,25,13,8,2507,2706,297822


In [296]:
## otherwise do this:
team_goalie_stats = goalies_individual.groupby(['teamAbbrev', 'season_id']).agg(
    team_savePct=pd.NamedAgg(
        column='i_savePct',
        aggfunc=lambda x: (
            (x * goalies_individual.loc[x.index, 'i_shotsAgainst']).sum()
            / goalies_individual.loc[x.index, 'i_shotsAgainst'].sum()
        )
    ),
    team_GAA=pd.NamedAgg(
        column='i_GAA',
        aggfunc=lambda x: (
            (x * goalies_individual.loc[x.index, 'i_timeOnIce']).sum()
            / goalies_individual.loc[x.index, 'i_timeOnIce'].sum()
        )
    ),
    team_wins=('i_wins', 'sum'),
    team_losses=('i_losses', 'sum'),
    team_shutouts=('i_shutouts', 'sum'),
    team_saves=('i_saves', 'sum'),
    team_shotsAgainst=('i_shotsAgainst', 'sum'),
).reset_index()


team_goalie_stats.head()

,teamAbbrev,season_id,team_savePct,team_GAA,team_wins,team_losses,team_shutouts,team_saves,team_shotsAgainst
0,ANA,20132014,0.915279,2.405251,54.0,20.0,6.0,2150.0,2349.0
1,ANA,20142015,0.909704,2.564931,51.0,24.0,4.0,2146.0,2359.0
2,ANA,20152016,0.918772,2.218657,46.0,25.0,8.0,2070.0,2253.0
3,ANA,20162017,0.920788,2.325711,46.0,23.0,8.0,2232.0,2424.0
4,ANA,20172018,0.926456,2.405461,44.0,25.0,8.0,2507.0,2706.0


In [297]:
team_goalie_stats = team_goalie_stats.rename(columns={'teamAbbrev':'tri_code'})
team_goalie_stats = team_goalie_stats[['season_id', 'tri_code', 'team_savePct', 'team_GAA']]
team_goalie_stats.head()

,season_id,tri_code,team_savePct,team_GAA
0,20132014,ANA,0.915279,2.405251
1,20142015,ANA,0.909704,2.564931
2,20152016,ANA,0.918772,2.218657
3,20162017,ANA,0.920788,2.325711
4,20172018,ANA,0.926456,2.405461


### Current Standings

In [298]:
# # today's standings
# base_url = "https://api-web.nhle.com/v1/standings/"
# current_date = pd.to_datetime(date.today())
# formatted_date = current_date.strftime("%Y-%m-%d")
# api_url = f"{base_url}{formatted_date}"

# response = requests.get(api_url)

# # Check if the request was successful (status code 200)
# if response.status_code != 200:
#     print(f"Request failed with status code {response.status_code}")
#     print(response.text[:200])  # optional debug peek
#     raise SystemExit  # or return / handle however you want

# # Safely parse JSON
# try:
#     json_data = response.json()
# except json.JSONDecodeError:
#     print("Could not decode JSON from standings endpoint:")
#     print(response.text[:200])
#     raise

# standings = json_data["standings"]
# standings_df = pd.DataFrame(standings)

# # Extract team names
# standings_df["team_name"] = standings_df["teamName"].apply(lambda x: x["default"])
# standings_df["tri_code"] = standings_df["teamAbbrev"].apply(lambda x: x["default"])
# standings_df["season_id"] = standings_df["seasonId"]
# standings_df["num_teams"] = 32

# standings_df = standings_df[
#     [
#         "conferenceName",
#         "team_name",
#         "tri_code",
#         "gamesPlayed",
#         "points",
#         "winPctg",
#         "pointPctg",
#         "regulationPlusOtWins",
#         "goalDifferential",
#         "goalFor",
#         "goalAgainst",
#         "homeGamesPlayed",
#         "homeWins",
#         "roadGamesPlayed",
#         "roadWins",
#         "l10GoalDifferential",
#         "seasonId",
#         # "clinchIndicator",
#     ]
# ]

# # Avoid divide-by-zero just in case
# standings_df["home_win_pctg"] = standings_df["homeWins"] / standings_df["homeGamesPlayed"].replace(0, np.nan)
# standings_df["road_win_pctg"] = standings_df["roadWins"] / standings_df["roadGamesPlayed"].replace(0, np.nan)

# points_per_game = standings_df["points"] / standings_df["gamesPlayed"].replace(0, np.nan)
# standings_df["gf_g"] = standings_df["goalFor"] / standings_df["gamesPlayed"].replace(0, np.nan)
# standings_df["ga_g"] = standings_df["goalAgainst"] / standings_df["gamesPlayed"].replace(0, np.nan)

# games_remaining = 82 - standings_df["gamesPlayed"]
# projected_final_points = standings_df["points"] + (games_remaining * points_per_game)
# standings_df["projected_points"] = round(projected_final_points)

# current_standings = standings_df


# current_standings

# if final_df is not None:
#     export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\current_standings.csv"
#     os.makedirs(os.path.dirname(export_path), exist_ok=True)
#     final_df.to_csv(export_path, index=False)
#     print("✅ Final dataset exported successfully!")
# else:
#     print("❌ No data returned.")
# current_standings.head()


### Specific Season Standings

In [299]:
# # season_start_dates = {
# #     "20252026": "2025-10-07",

# # }

# # # season_end_dates = {

# # #     "20252026": "2026-04-16",
# # # }
# # season_end_dates = {

# #     "20252026": "2026-11-23",
# # } 

# # def get_standings_snapshots():
# #     base_url = "https://api-web.nhle.com/v1/standings/"
# #     all_rows = []

# #     for season in season_start_dates:
# #         start_date = pd.to_datetime(season_start_dates[season])
# #         end_date = pd.to_datetime(season_end_dates[season])

# #         print(f"Processing season {season}...")

# #         for check_date in pd.date_range(start=start_date, end=end_date, freq='1D'): #date frequency - every 3 days
# #             url = f"{base_url}{check_date.strftime('%Y-%m-%d')}"
# #             try:
# #                 response = requests.get(url)
# #                 if response.status_code != 200:
# #                     print(f"Failed request: {url}")
# #                     continue

# #                 standings_data = response.json()["standings"]
# #                 for team in standings_data:
# #                     row = {
# #                         "date": check_date.date(),
# #                         "season": season,
# #                         "team": team.get("teamName", {}).get("default", None),
# #                         "tri_code": team.get("teamAbbrev", {}).get("default", None),
# #                         "gp": team.get("gamesPlayed", 0),
# #                         "conference_name": team.get("conferenceName", None),
# #                         "conference_sequence": team.get("conferenceSequence", 0),
# #                         "division_name": team.get("divisionName", None),
# #                         "division_sequence": team.get("divisionSequence", 0),
# #                         "pointPctg":team.get("pointPctg", 0),
# #                         "points":team.get("points", 0),
# #                     }

# #                     all_rows.append(row)

# #             except Exception as e:
# #                 print(f"Error on {check_date.date()} ({season}): {e}")

# #     standings_df = pd.DataFrame(all_rows)
# #     return standings_df
# # df = get_standings_snapshots()
# standings_2425 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\history_final_standings.csv")
# standings_2425.columns


### Standings Snap Shots 

In [300]:

# season_start_dates = {
    # "20002001": "2000-10-04",
    # "20012002": "2001-10-03",
    # "20022003": "2002-10-09",
    # "20032004": "2003-10-08",
    # "20052006": "2005-10-05",
    # "20062007": "2006-10-04",
    # "20072008": "2007-09-29",
    # "20082009": "2008-10-04",
    # "20092010": "2009-10-01",
    # "20102011": "2010-10-07",
    # "20112012": "2011-10-06",
    # "20122013": "2013-01-19",
    # "20132014": "2013-10-01",
    # "20142015": "2014-10-08",
    # "20152016": "2015-10-07",
    # "20162017": "2016-10-12",
    # "20172018": "2017-10-04",
    # "20182019": "2018-10-04",
    # "20192020": "2019-10-02",
    # "20202021": "2021-01-13",
    # "20212022": "2021-10-12",
    # "20222023": "2022-10-11",
    # "20232024": "2023-10-11",
    # "20242025": "2024-10-04"
# }

# Season End Dates

# season_end_dates = {
    # "20002001": "2001-04-08",
    # "20012002": "2002-04-14",
    # "20022003": "2003-04-06",
    # "20032004": "2004-04-04",
    # "20052006": "2006-04-18",
    # "20062007": "2007-04-08",
    # "20072008": "2008-04-06",
    # "20082009": "2009-04-12",
    # "20092010": "2010-04-11",
    # "20102011": "2011-04-10",
    # "20112012": "2012-04-07",
    # "20122013": "2013-04-27",
    # "20132014": "2014-04-13",
    # "20142015": "2015-04-11",
    # "20152016": "2016-04-10",
    # "20162017": "2017-04-09",
    # "20172018": "2018-04-08",
    # "20182019": "2019-04-06",
    # "20192020": "2020-03-10",
    # "20202021": "2021-05-19",
    # "20212022": "2022-04-29",
    # "20222023": "2023-04-13",
    # "20232024": "2024-04-18",
    # "20242025": "2025-04-17"
# }

# def safe_get(url, retries=5, base_sleep=0.25):
#     """
#     Safely GET an NHL API URL with:
#     - 429 exponential backoff
#     - throttle delay
#     - retry on transient errors
#     """
#     for attempt in range(retries):
#         try:
#             response = requests.get(url)

#             # --- Success ---
#             if response.status_code == 200:
#                 time.sleep(base_sleep)   # throttle to avoid 429
#                 return response

#             # --- Handle 429 Too Many Requests ---
#             if response.status_code == 429:
#                 wait = 2 ** attempt
#                 print(f"[429] Rate limit hit — waiting {wait} sec...")
#                 time.sleep(wait)
#                 continue

#             # --- Other errors ---
#             print(f"[{response.status_code}] Error hitting {url}")
#             return None

#         except requests.exceptions.RequestException as e:
#             wait = 2 ** attempt
#             print(f"[Network error] {e} — retrying in {wait} sec...")
#             time.sleep(wait)

#     print("[ERROR] Max retries reached for:", url)
#     return None


# # ============================================================
# #   MAIN SNAPSHOT SCRAPER FUNCTION
# # ============================================================

# def get_standings_snapshots():
#     base_url = "https://api-web.nhle.com/v1/standings/"
#     all_rows = []

#     for season in season_start_dates:
#         start_date = pd.to_datetime(season_start_dates[season])
#         end_date = pd.to_datetime(season_end_dates[season])

#         print(f"\n=== Processing season {season} ===")

#         for check_date in pd.date_range(start=start_date, end=end_date, freq='10D'):

#             url = f"{base_url}{check_date.strftime('%Y-%m-%d')}"
#             response = safe_get(url)

#             if response is None:
#                 print(f"Skipping date {check_date.date()} — no response")
#                 continue

#             try:
#                 standings_data = response.json().get("standings", [])
#             except ValueError:
#                 print(f"Invalid JSON on {check_date.date()}: {url}")
#                 continue

#             for team in standings_data:
#                 row = {
#                     "date": check_date.date(),
#                     "season": season,
#                     "team": team.get("teamName", {}).get("default", None),
#                     "tri_code": team.get("teamAbbrev", {}).get("default", None),
#                     "gp": team.get("gamesPlayed", 0),
#                     "conference_name": team.get("conferenceName", None),
#                     "conference_sequence": team.get("conferenceSequence", 0),
#                     "division_name": team.get("divisionName", None),
#                     "division_sequence": team.get("divisionSequence", 0),
#                     "pointPctg": team.get("pointPctg", 0),
#                     "points": team.get("points", 0),
#                     "l10GamesPlayed": team.get("l10GamesPlayed", 0),
#                     "l10GoalDifferential": team.get("l10GoalDifferential", 0),
#                     "l10GoalsAgainst": team.get("l10GoalsAgainst", 0),
#                     "l10GoalsFor": team.get("l10GoalsFor", 0),
#                     "l10Losses": team.get("l10Losses", 0),
#                     "l10OtLosses": team.get("l10OtLosses", 0),
#                     "l10Points": team.get("l10Points", 0),
#                     "l10RegulationPlusOtWins": team.get("l10RegulationPlusOtWins", 0),
#                     "l10RegulationWins": team.get("l10RegulationWins", 0),
#                     "l10Ties": team.get("l10Ties", 0),
#                     "l10Wins": team.get("l10Wins", 0),
#                     "conferenceL10Sequence": team.get("conferenceL10Sequence", 0),
#                     "divisionL10Sequence": team.get("divisionL10Sequence", 0),
#                     "divisionSequence": team.get("divisionSequence", 0),
#                     "regulationPlusOtWins": team.get("regulationPlusOtWins", 0),
#                     "goal_diff": team.get("goalDifferential", 0),
#                     "goals_for": team.get("goalFor", 0),
#                     "goals_against": team.get("goalAgainst", 0),
#                     "home_wins": team.get("homeWins", 0),
#                     "road_wins": team.get("roadWins", 0),
#                 }

#                 all_rows.append(row)

#     return pd.DataFrame(all_rows)
# df = get_standings_snapshots()

# def classify_teams(df):
#     final_snapshots = df.sort_values('date').groupby(['season', 'tri_code']).tail(1)

#     # Rank within conference
#     final_snapshots['conference_rank'] = final_snapshots.groupby(['season', 'conference_name'])['points'] \
#         .rank(ascending=False, method='min')

#     def classify(rank):
#         if rank <= 3:
#             return 'Top'
#         elif rank <= 6:
#             return 'Middle'
#         elif rank <= 9:
#             return 'Bubble'
#         else:
#             return 'Out'

#     final_snapshots['team_class'] = final_snapshots['conference_rank'].apply(classify)

#     return final_snapshots[['season', 'tri_code', 'team_class']]

# def calculate_pctg_change(df):
#     df = df.sort_values(['season', 'tri_code', 'date'])
#     df['prev_pctg'] = df.groupby(['season', 'tri_code'])['pointPctg'].shift(1)
#     df['pctg_change'] = df['pointPctg'] - df['prev_pctg']
#     return df

# # --- Step 4: Aggregate by Class & Date ---
# def aggregate_by_class_and_date(df):
#     agg = df.groupby(['season', 'date', 'team_class'])['pctg_change'].mean().reset_index()
#     return agg

# # --- Main Pipeline ---
# def run_analysis():
#     df = get_standings_snapshots()
#     team_classes = classify_teams(df)
#     df = df.merge(team_classes, on=['season', 'tri_code'], how='left')
#     df = calculate_pctg_change(df)
#     agg_df = aggregate_by_class_and_date(df)
#     return df, agg_df

# # Run everything
# snapshots_df, impact_df = run_analysis()


# snapshots_df = snapshots_df[['date', 'season', 'team', 'tri_code', 'team_class',  'conference_name', 'conference_sequence']]

# def calculate_pctg_change(snapshots_df):
#     snapshots_df = snapshots_df.sort_values(['season', 'tri_code', 'date'])
#     snapshots_df['prev_rank'] = snapshots_df.groupby(['season', 'tri_code'])['conference_sequence'].shift(1)
#     snapshots_df['rank_change'] = snapshots_df['prev_rank']- snapshots_df['conference_sequence']
#     snapshots_df['prev_rank']=snapshots_df['prev_rank'].fillna(0)
#     snapshots_df['rank_change']=snapshots_df['rank_change'].fillna(0)
#     # Ensure 'date' column is in datetime format
#     snapshots_df['date'] = pd.to_datetime(snapshots_df['date'], errors='coerce')
#     snapshots_df['month'] = snapshots_df['date'].dt.month

#     return snapshots_df

# snapshots_df = calculate_pctg_change(snapshots_df)


# # Group by season, team, tri_code, and month, then calculate the mean of conference_sequence
# snapshots_df = snapshots_df.groupby(['date','season', 'team', 'tri_code', 'month'])['conference_sequence'].mean()

# # Convert the result back to a DataFrame
# snapshots_df = pd.DataFrame(snapshots_df).reset_index()
# snapshots_df['conference_sequence'] = round(snapshots_df['conference_sequence'],2)



# # Display the final DataFrame
# def calculate_month_change(snapshots_df):
#     # Sort values by season, tri_code, and month (adjust to ensure months are sorted correctly)
#     snapshots_df['month'] = snapshots_df['month'].apply(lambda x: x if x >= 10 else x + 12)  # Adjust months
#     snapshots_df = snapshots_df.sort_values(['season', 'tri_code', 'month'])
    
#     # Calculate previous rank by shifting the 'conference_sequence' within each team-season group
#     snapshots_df['prev_rank'] = snapshots_df.groupby(['season', 'tri_code'])['conference_sequence'].shift(1)
    
#     # Calculate the rank change
#     snapshots_df['rank_change'] = snapshots_df['prev_rank'] - snapshots_df['conference_sequence']
    
#     # Fill NaN values (for first month) with 0
#     snapshots_df['prev_rank'] = snapshots_df['prev_rank'].fillna(0)
#     snapshots_df['rank_change'] = snapshots_df['rank_change'].fillna(0)
    
#     # Reset the month adjustment to original range (10-12, 1-4)
#     snapshots_df['month'] = snapshots_df['month'].apply(lambda x: x if x <= 12 else x - 12)
    
#     return snapshots_df

# # Run the function to calculate the month change
# snapshots_df = calculate_month_change(snapshots_df)

## append
# dfs = [history_snaps,snap_shot_2425]

# expected_cols = ['date', 'season', 'team', 'tri_code', 'gamesPlayed', 'l10GamesPlayed',
#        'l10GoalDifferential', 'l10GoalsAgainst', 'l10GoalsFor', 'l10Losses',
#        'l10OtLosses', 'l10Points', 'l10RegulationPlusOtWins',
#        'l10RegulationWins', 'l10Ties', 'l10Wins', 'pointPctg', 'points',
#        'conferenceL10Sequence', 'conferenceName', 'conferenceSequence',
#        'divisionL10Sequence', 'divisionName', 'divisionSequence',
#        'regulationPlusOtWins', 'goal_diff', 'goals_for', 'goals_against',
#        'home_wins', 'road_wins']

# aligned_dfs = [df.reindex(columns=expected_cols) for df in dfs]
# standing_snapshots = pd.concat(aligned_dfs, ignore_index=True)


# def classify_teams(df):
#     final_snapshots = df.sort_values('date').groupby(['season', 'tri_code']).tail(1)

#     # Rank within conference
#     final_snapshots['conference_rank'] = final_snapshots.groupby(['season', 'conference_name'])['points'] \
#         .rank(ascending=False, method='min')

#     def classify(rank):
#         if rank <= 3:
#             return 'Top'
#         elif rank <= 6:
#             return 'Middle'
#         elif rank <= 9:
#             return 'Bubble'
#         else:
#             return 'Out'

#     final_snapshots['team_class'] = final_snapshots['conference_rank'].apply(classify)

#     return final_snapshots[['season', 'tri_code', 'team_class']]

# def calculate_pctg_change(df):
#     df = df.sort_values(['season', 'tri_code', 'date'])
#     df['prev_pctg'] = df.groupby(['season', 'tri_code'])['pointPctg'].shift(1)
#     df['pctg_change'] = df['pointPctg'] - df['prev_pctg']
#     return df

# # --- Step 4: Aggregate by Class & Date ---
# def aggregate_by_class_and_date(df):
#     agg = df.groupby(['season', 'date', 'team_class'])['pctg_change'].mean().reset_index()
#     return agg

# # --- Main Pipeline ---
# def run_analysis():
#     df = get_standings_snapshots()
#     team_classes = classify_teams(df)
#     df = df.merge(team_classes, on=['season', 'tri_code'], how='left')
#     df = calculate_pctg_change(df)
#     agg_df = aggregate_by_class_and_date(df)
#     return df, agg_df

# # Run everything
# snapshots_df, impact_df = run_analysis()



# snapshots_df.head()

### Clinch History Dates

In [ ]:
# # # season_end_dates = {
# # #        "20132014": "2014-04-13",
# # # }    

# # # season_start_dates = {
# # #      "20192020": "2019-10-02",
# # # }


# base_url = "https://api-web.nhle.com/v1/standings/"
# all_clinch_rows = []

# # Hard-restrict to only specific season
# target_season = "20192020"

# # Use your explicit start/end dates
# start_date = pd.to_datetime("2019-10-02")
# end_date = pd.to_datetime("2020-03-01")

# print(f"Processing season {target_season}")

# today = end_date
# last_ran_date = start_date

# team_status = {}

# # Loop from Oct 2 → Mar 12
# for check_date in pd.date_range(start=start_date, end=end_date):

#     url = f"{base_url}{check_date.strftime('%Y-%m-%d')}"

#     try:
#         response = requests.get(url)
#         if response.status_code != 200:
#             continue

#         standings_data = response.json()["standings"]

#         # Conferences
#         east_teams = [t for t in standings_data if t["conferenceName"] == "Eastern"]
#         west_teams = [t for t in standings_data if t["conferenceName"] == "Western"]

#         for conference_teams in [east_teams, west_teams]:

#             # 8th place team (cutoff)
#             eighth_place_team = next(
#                 (t for t in conference_teams if t["conferenceSequence"] == 8),
#                 None
#             )
#             cutoff_points = eighth_place_team["points"] if eighth_place_team else 0

#             for team in conference_teams:
#                 tri_code = team["teamAbbrev"]["default"]
#                 games_played = team["gamesPlayed"]
#                 points = team["points"]
#                 clinch = team.get("clinchIndicator", None)

#                 if tri_code not in team_status:
#                     team_status[tri_code] = {
#                         "last_ran_date": last_ran_date.date(),
#                         "clinch_indicator": False,
#                         "tri_code": tri_code,
#                         "clinch_type": None,
#                         "clinch_date": None,
#                         "season_end": today.date(),
#                         "season": target_season
#                     }

#                 # If clinched
#                 if clinch and team_status[tri_code]["clinch_type"] is None:
#                     team_status[tri_code]["clinch_type"] = clinch
#                     team_status[tri_code]["clinch_indicator"] = True
#                     team_status[tri_code]["clinch_date"] = check_date.date()
#                     continue

#                 # Elimination logic
#                 total_games = 82
#                 remaining_games = total_games - games_played
#                 max_points = points + (remaining_games * 2)

#                 if (team_status[tri_code]["clinch_type"] is None 
#                     and max_points < cutoff_points):
#                     team_status[tri_code]["clinch_type"] = "e"
#                     team_status[tri_code]["clinch_indicator"] = True
#                     team_status[tri_code]["clinch_date"] = check_date.date()

#     except Exception as e:
#         print(f"Error on {check_date.date()} for {target_season}: {e}")

# # Backfill any team missing a clinch marker
# for tri_code, data in team_status.items():
#     if data["clinch_type"] is None:
#         data["clinch_type"] = "e"
#         data["clinch_indicator"] = True
#         data["clinch_date"] = end_date.date()
#     all_clinch_rows.append(data)

# clinch_history_df = pd.DataFrame(all_clinch_rows)
# clinch_history_df = clinch_history_df[
#     ['last_ran_date', 'clinch_indicator', 'tri_code',
#         'clinch_type', 'clinch_date', 'season_end', 'season']
# ].sort_values(by=["season", "clinch_date", "tri_code"]).reset_index(drop=True)



# # Define export path
# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\history_clinch_1920.csv" # -- update path

# # Ensure the folder exists before writing the file
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # # Export the data to CSV
# clinch_history_df.to_csv(export_path, index=False)

# clinch_history_df.head()


Processing season 20192020


,last_ran_date,clinch_indicator,tri_code,clinch_type,clinch_date,season_end,season
0,2019-10-02,True,ANA,e,2020-03-01,2020-03-01,20192020
1,2019-10-02,True,ARI,e,2020-03-01,2020-03-01,20192020
2,2019-10-02,True,BOS,e,2020-03-01,2020-03-01,20192020
3,2019-10-02,True,BUF,e,2020-03-01,2020-03-01,20192020
4,2019-10-02,True,CAR,e,2020-03-01,2020-03-01,20192020


In [302]:
# # Ensure last_ran_date is a datetime object
# last_ran_date = pd.to_datetime(last_ran_date)

# # === TODAY'S DATE ===
# today = pd.to_datetime(date.today())

# # === TRACK FIRST-TIME CLINCHES ===
# seen_teams = set(clinch_history["tri_code"]) if 'tri_code' in clinch_history.columns else set()
# first_clinch = {}
# base_url = "https://api-web.nhle.com/v1/standings/"

# for check_date in pd.date_range(start=last_ran_date + timedelta(days=1), end=today):
#     url = f"{base_url}{check_date.strftime('%Y-%m-%d')}"
#     try:
#         response = requests.get(url)
#         if response.status_code != 200:
#             continue
#         standings_data = response.json()["standings"]
#         for team in standings_data:
#             tri_code = team["teamAbbrev"]["default"]
#             clinch = team.get("clinchIndicator")

#             if clinch and tri_code not in seen_teams and tri_code not in first_clinch:
#                 first_clinch[tri_code] = {
#                     "last_ran_date": last_ran_date,
#                     "clinch_indicator": True,
#                     "tri_code": tri_code,
#                     "clinch_type": clinch,
#                     "clinch_date": check_date,
#                     "today": today
#                 }
#     except Exception as e:
#         print(f"Error on {check_date}: {e}")

# # === APPEND NEW CLINCHES ===
# if first_clinch:
#     new_df = pd.DataFrame(first_clinch.values())
#     clinch_history = pd.concat([clinch_history, new_df], ignore_index=True)

# # === REMOVE INITIAL PLACEHOLDER ROW IF CLINCHES HAVE BEEN LOGGED ===
# if 'tri_code' in clinch_history.columns:
#     clinch_history = clinch_history[~(clinch_history['clinch_indicator'] == False) | clinch_history['tri_code'].notna()]

# # === UPDATE last_ran_date to today ===
# clinch_history['last_ran_date'] = today

# # Strip time from date columns
# clinch_history['last_ran_date'] = pd.to_datetime(clinch_history['last_ran_date']).dt.date
# clinch_history['today'] = pd.to_datetime(clinch_history['today'], errors='coerce').dt.date

# clinch_history['clinch_date'] = pd.to_datetime(clinch_history['clinch_date'], errors='coerce').dt.date

# # === SAVE FINAL FILE ===
# clinch_history.to_csv(history_path, index=False)
# clinch_history.head()


In [303]:
# clinch_history_full = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\full_clinch_history.csv")

# clinch_history_full['clinch_date'] = pd.to_datetime(
#     clinch_history_full['clinch_date'], errors='coerce'
# ).dt.strftime('%Y-%m-%d')
# clinch_history_full.tail()


### Rush Chances

In [304]:
game_plays.tail()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
4742758,20242025,2024021038,526,19:37:00,00:23:00,525,takeaway,289,1,24,-93.0,-7.0,D,<NA>,<NA>,None,<NA>,<NA>,<NA>,8484153,None,<NA>,<NA>,5 on 5,in net,1177,59,24,20242025,2024021038
4742759,20242025,2024021038,519,19:42:00,00:18:00,508,blocked-shot,290,1,24,-64.0,-12.0,D,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,blocked,8483431,<NA>,5 on 5,in net,1182,59,24,20242025,2024021038
4742760,20242025,2024021038,202,19:43:00,00:17:00,507,missed-shot,291,1,24,-92.0,-18.0,O,<NA>,<NA>,wrist,8478872,<NA>,<NA>,<NA>,failed-bank-attempt,<NA>,<NA>,5 on 5,in net,1183,59,24,20242025,2024021038
4742761,20242025,2024021038,26,19:44:00,00:16:00,516,stoppage,292,1,<NA>,NaN,NaN,None,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,goalie-stopped-after-sog,<NA>,<NA>,5 on 5,in net,1184,59,24,20242025,2024021038
4742762,20242025,2024021038,73,19:44:00,00:16:00,502,faceoff,294,1,59,-69.0,-22.0,D,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,1184,59,24,20242025,2024021038


In [305]:
# --------------------------------------
# 1. Define event groups
# --------------------------------------
shot_events = {
    "shot-on-goal", 
    "missed-shot", 
    "goal", 
    "blocked-shot", 
    "failed-shot-attempt"
}

transition_events = {
    "giveaway",
    "takeaway",
    "hit"
}

stoppage_events = {
    "faceoff",
    "stoppage",
    "period-end",
    "game-end",
    "penalty",
    "delayed-penalty",
}

valid_prev_events = shot_events | transition_events | stoppage_events

# --------------------------------------
# 2. Filter to only events we care about
# --------------------------------------
mask = game_plays["typeDescKey"].isin(valid_prev_events)
valid_plays = game_plays.loc[mask].copy()

# Sort by game + time
valid_plays = valid_plays.sort_values(["game_id", "sortOrder"])

# --------------------------------------
# 3. Build "previous event" context
# --------------------------------------
grouped = valid_plays.groupby("game_id", sort=False)

valid_plays["prev_event_time"] = grouped["game_in_seconds"].shift(1)
valid_plays["prev_event_type"] = grouped["typeDescKey"].shift(1)
valid_plays["prev_zone"]       = grouped["zone_code"].shift(1)
valid_plays["prev_team_id"]    = grouped["team_id"].shift(1)

valid_plays["prev_event_was_stoppage"] = valid_plays["prev_event_type"].isin(stoppage_events)
valid_plays["time_diff"] = valid_plays["game_in_seconds"] - valid_plays["prev_event_time"]
# --------------------------------------
# 4. Rush shot condition
# --------------------------------------
is_shot = valid_plays["typeDescKey"].isin(shot_events)

# Time window: allow 0–5 seconds
quick = valid_plays["time_diff"].between(0, 5, inclusive="both")

# Possession change: previous team != shooting team
possession_change = (
    valid_plays["prev_team_id"].notna() &
    (valid_plays["prev_team_id"] != valid_plays["team_id"])
)

# Same team, but coming from DZ or NZ
same_team_dz_nz = (
    (valid_plays["prev_team_id"] == valid_plays["team_id"]) &
    (valid_plays["prev_zone"].isin(["D", "N"]))
)

no_stoppage = ~valid_plays["prev_event_was_stoppage"]

rush_mask = (
    is_shot &
    quick &
    no_stoppage &
    (possession_change | same_team_dz_nz)
)

rush_df = valid_plays.loc[rush_mask].copy()


rush_df["rush_shot"] = 1


In [306]:
# --------------------------------------
# 5. Rush shots FOR (team taking the rush shot)
# --------------------------------------
rush_shots_for = (
    rush_df
    .groupby(["season_id", "team_id"])
    .size()
    .reset_index(name="rush_shots_for")
)

# --------------------------------------
# 6. Rush shots AGAINST (team whose previous event enabled the rush)
# --------------------------------------
rush_shots_against = (
    rush_df
    .groupby(["season_id", "prev_team_id"])
    .size()
    .reset_index(name="rush_shots_against")
    .rename(columns={"prev_team_id": "team_id"})
)

# --------------------------------------
# 7. Merge FOR + AGAINST
# --------------------------------------
rush_stats = rush_shots_for.merge(
    rush_shots_against,
    on=["season_id", "team_id"],
    how="outer"
)

rush_stats = rush_stats.fillna(0)

rush_stats["rush_shots_for"] = rush_stats["rush_shots_for"].astype(int)
rush_stats["rush_shots_against"] = rush_stats["rush_shots_against"].astype(int)

rush_stats["rush_shot_differential"] = (
    rush_stats["rush_shots_for"] - rush_stats["rush_shots_against"]
)

rush_stats = (
    rush_stats
    .sort_values(["season_id", "rush_shot_differential"], ascending=[True, False])
    .reset_index(drop=True)
)

rush_stats.head()


,season_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential
0,20132014,16,511,395,116
1,20132014,28,529,458,71
2,20132014,1,357,298,59
3,20132014,23,458,409,49
4,20132014,17,417,372,45


In [307]:
rush_stats = rush_stats.merge(teams, on = "team_id", how="left")
rush_stats.head()

,season_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential,tri_code
0,20132014,16,511,395,116,CHI
1,20132014,28,529,458,71,SJS
2,20132014,1,357,298,59,NJD
3,20132014,23,458,409,49,VAN
4,20132014,17,417,372,45,DET


In [308]:
rush_df_abbrev = rush_df[['game_id', 'eventId', 'game_in_seconds', 'prev_event_time', 'prev_zone', 'prev_event_type', 'prev_team_id', 'prev_event_was_stoppage', 'time_diff', 'rush_shot']]
rush_df_abbrev.head()

,game_id,eventId,game_in_seconds,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
5,2013020001,55,105,100.0,O,hit,8,False,5.0,1
22,2013020001,63,178,176.0,D,hit,8,False,2.0,1
37,2013020001,71,225,224.0,D,giveaway,10,False,1.0,1
49,2013020001,78,320,316.0,D,blocked-shot,10,False,4.0,1
80,2013020001,97,692,689.0,D,hit,10,False,3.0,1


### All Shot Attempts

In [309]:
all_shots_made = game_plays.copy()


In [310]:
all_shots_made = all_shots_made[all_shots_made["typeCode"].isin([505, 506, 507, 508, 537])]
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
2,20132014,2013020001,53,01:14:00,18:46:00,507,missed-shot,13,1,10,35.0,-28.0,O,<NA>,<NA>,wrist,8471679,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,74,8,10,20132014,2013020001
3,20132014,2013020001,54,01:36:00,18:24:00,506,shot-on-goal,18,1,8,-82.0,-22.0,O,<NA>,<NA>,wrist,8473503,0,1,<NA>,None,<NA>,<NA>,5 on 5,in net,96,8,10,20132014,2013020001
5,20132014,2013020001,55,01:45:00,18:15:00,508,blocked-shot,21,1,10,57.0,-31.0,D,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,None,8475797,<NA>,5 on 5,in net,105,8,10,20132014,2013020001
8,20132014,2013020001,57,01:52:00,18:08:00,508,blocked-shot,26,1,10,37.0,-31.0,D,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,None,8468635,<NA>,5 on 5,in net,112,8,10,20132014,2013020001
15,20132014,2013020001,59,02:24:00,17:36:00,506,shot-on-goal,36,1,8,-44.0,-5.0,O,<NA>,<NA>,wrist,8473503,0,2,<NA>,None,<NA>,<NA>,5 on 5,in net,144,8,10,20132014,2013020001


In [311]:
all_shots_made["goalie"] = all_shots_made["goalie"].fillna(0)


In [312]:
#add in the rush shot attempts
all_shots_made = all_shots_made.merge(rush_df_abbrev, on=['game_id', 'eventId','game_in_seconds'], how = 'left')
all_shots_made['rush_shot']= all_shots_made['rush_shot'].fillna(0)
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
0,20132014,2013020001,53,01:14:00,18:46:00,507,missed-shot,13,1,10,35.0,-28.0,O,<NA>,<NA>,wrist,8471679,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,74,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
1,20132014,2013020001,54,01:36:00,18:24:00,506,shot-on-goal,18,1,8,-82.0,-22.0,O,<NA>,<NA>,wrist,8473503,0,1,<NA>,None,<NA>,<NA>,5 on 5,in net,96,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
2,20132014,2013020001,55,01:45:00,18:15:00,508,blocked-shot,21,1,10,57.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8475797,<NA>,5 on 5,in net,105,8,10,20132014,2013020001,100.0,O,hit,8,False,5.0,1.0
3,20132014,2013020001,57,01:52:00,18:08:00,508,blocked-shot,26,1,10,37.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8468635,<NA>,5 on 5,in net,112,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
4,20132014,2013020001,59,02:24:00,17:36:00,506,shot-on-goal,36,1,8,-44.0,-5.0,O,<NA>,<NA>,wrist,8473503,0,2,<NA>,None,<NA>,<NA>,5 on 5,in net,144,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0


In [313]:
# Identify Corsi shots - those only in the O Zone

all_shots_made['corsi'] = np.where(all_shots_made['zone_code']== "O", 1, 0)
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi
0,20132014,2013020001,53,01:14:00,18:46:00,507,missed-shot,13,1,10,35.0,-28.0,O,<NA>,<NA>,wrist,8471679,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,74,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1
1,20132014,2013020001,54,01:36:00,18:24:00,506,shot-on-goal,18,1,8,-82.0,-22.0,O,<NA>,<NA>,wrist,8473503,0,1,<NA>,None,<NA>,<NA>,5 on 5,in net,96,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1
2,20132014,2013020001,55,01:45:00,18:15:00,508,blocked-shot,21,1,10,57.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8475797,<NA>,5 on 5,in net,105,8,10,20132014,2013020001,100.0,O,hit,8,False,5.0,1.0,0
3,20132014,2013020001,57,01:52:00,18:08:00,508,blocked-shot,26,1,10,37.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8468635,<NA>,5 on 5,in net,112,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0
4,20132014,2013020001,59,02:24:00,17:36:00,506,shot-on-goal,36,1,8,-44.0,-5.0,O,<NA>,<NA>,wrist,8473503,0,2,<NA>,None,<NA>,<NA>,5 on 5,in net,144,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1


### SH%
- SH%: Percentage of Shots for that team that were Goals. GF*100/SF
- SV%: Percentage of Shots against that team that were not Goals. 100-(GA*100/SA)

In [314]:
sh_percent = all_shots_made[['game_id', 'season_id', 'team_id', 'typeDescKey']].copy()

# Create shot-type indicator columns
sh_percent['block'] = np.where(sh_percent['typeDescKey'] == 'blocked-shot', 1, 0)
sh_percent['goal'] = np.where(sh_percent['typeDescKey'] == 'goal', 1, 0)

# A "shot" is any attempt that is NOT blocked
# (blocked shots never reach the net, so they do not count toward SOG)
sh_percent['shots'] = np.where(sh_percent['block'] == 1, 0, 1)

# Group by season + team
sh_percent = (
    sh_percent
        .groupby(['season_id', 'team_id'], as_index=False)
        .agg(
            shots=('shots', 'sum'),
            goals=('goal', 'sum'),
        )
)

# Safe shooting percentage (avoids division-by-zero)
sh_percent['shot_pct'] = (
    sh_percent.apply(
        lambda row: round((row['goals'] * 100) / row['shots'], 2)
        if row['shots'] > 0 else 0,
        axis=1
    )
)


sh_percent.head()

,season_id,team_id,shots,goals,shot_pct
0,20132014,1,3077,201,6.53
1,20132014,2,3525,239,6.78
2,20132014,3,3738,223,5.97
3,20132014,4,3534,244,6.90
4,20132014,5,3351,253,7.55


In [315]:

## shots made against
shots_against = all_shots_made[['game_id','season_id', 'team_id', 'typeDescKey']].copy()

# Set up columns
shots_against['block'] = np.where(shots_against['typeDescKey'] == 'blocked-shot', 1, 0)
shots_against['goal'] = np.where(shots_against['typeDescKey'] == "goal", 1, 0)
shots_against['shot'] = np.where(shots_against['typeDescKey'] != "goal", 1, 0)
shots_against['shots'] = np.where(shots_against['block'] == 1, 0, 1)


# ----------------------------------------------------
# FIXED OPPONENT MAP (NO APPLY/STACK ERRORS)
# ----------------------------------------------------

# Extract the two team_ids per game
team_pairs = (
    shots_against[['game_id', 'team_id']]
    .drop_duplicates()
    .groupby('game_id')['team_id']
    .apply(list)
)

# Build opponent mapping as a dict-of-dicts
opponent_map = {}

for game_id, ids in team_pairs.items():
    if len(ids) == 2:
        opponent_map[game_id] = {
            ids[0]: ids[1],
            ids[1]: ids[0]
        }
    else:
        # fallback in case of weird data
        opponent_map[game_id] = {}


# Assign opponent team for each shot attempt
def get_opponent(row):
    lookup = opponent_map.get(row['game_id'], {})
    return lookup.get(row['team_id'])

shots_against['opponent_team_id'] = shots_against.apply(get_opponent, axis=1)

shots_against.head()

,game_id,season_id,team_id,typeDescKey,block,goal,shot,shots,opponent_team_id
0,2013020001,20132014,10,missed-shot,0,0,1,1,8
1,2013020001,20132014,8,shot-on-goal,0,0,1,1,10
2,2013020001,20132014,10,blocked-shot,1,0,1,0,8
3,2013020001,20132014,10,blocked-shot,1,0,1,0,8
4,2013020001,20132014,8,shot-on-goal,0,0,1,1,10


In [316]:
# ===============================
# Opponent / Against Shot Metrics
# ===============================

# Group by season + opponent team ID
against = (
    shots_against
        .groupby(['season_id', 'opponent_team_id'], as_index=False)
        .agg(
            shots_against=('shots', 'sum'),
            goals_against=('goal', 'sum')
        )
)

# Rename opponent_team_id → team_id (so it's consistent with your "for" dataset)
against = against.rename(columns={'opponent_team_id': 'team_id'})

# Safe shot percentage against
# (prevents division-by-zero warnings if a team allows 0 shots)
against['shot_pct_against'] = (
    against.apply(
        lambda row: round((row['goals_against'] * 100) / row['shots_against'], 2)
        if row['shots_against'] > 0 else 0,
        axis=1
    )
)

# Preview
against.head()


,season_id,team_id,shots_against,goals_against,shot_pct_against
0,20132014,1,2845,216,7.59
1,20132014,2,3541,279,7.88
2,20132014,3,3338,199,5.96
3,20132014,4,3463,244,7.05
4,20132014,5,3374,211,6.25


In [317]:

# Merge both DataFrames on team_id
shooting_stats = sh_percent.merge(against, on=['team_id','season_id'], how='left')

# Calculate Save Percentage (SV%)
shooting_stats['sv_pct'] = 100 - ((shooting_stats['goals_against'] * 100) / shooting_stats['shots_against'])

# Optional: round the SV% column
shooting_stats['sv_pct'] = shooting_stats['sv_pct'].round(2)

# Done!
shooting_stats.head()


,season_id,team_id,shots,goals,shot_pct,shots_against,goals_against,shot_pct_against,sv_pct
0,20132014,1,3077,201,6.53,2845,216,7.59,92.41
1,20132014,2,3525,239,6.78,3541,279,7.88,92.12
2,20132014,3,3738,223,5.97,3338,199,5.96,94.04
3,20132014,4,3534,244,6.90,3463,244,7.05,92.95
4,20132014,5,3351,253,7.55,3374,211,6.25,93.75


In [318]:
#PDO - Shooting percentage plus save percentage. (GF/SF)+(GA/SA)
shooting_stats['pdo'] = shooting_stats['shot_pct'] + shooting_stats['sv_pct']
shooting_stats = shooting_stats[['team_id', 'season_id', 'shot_pct', 'shots', 'shots_against', 'shot_pct_against', 'sv_pct', 'pdo']]
shooting_stats.head()

,team_id,season_id,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo
0,1,20132014,6.53,3077,2845,7.59,92.41,98.94
1,2,20132014,6.78,3525,3541,7.88,92.12,98.90
2,3,20132014,5.97,3738,3338,5.96,94.04,100.01
3,4,20132014,6.90,3534,3463,7.05,92.95,99.85
4,5,20132014,7.55,3351,3374,6.25,93.75,101.30


### Danger Zones

In [319]:
# Danger zone

# Define conditions based on the corrected MD and HD coordinates

# High Danger Zone (HD) - Right side of the rink
hd_condition_right = (
    (all_shots_made["x_coord"] >= 66.5)
    & (all_shots_made["x_coord"] <= 89)
    & (all_shots_made["y_coord"] >= -7)
    & (all_shots_made["y_coord"] <= 7)
)

# High Danger Zone (HD) - Left side of the rink (mirrored)
hd_condition_left = (
    (all_shots_made["x_coord"] >= -89)
    & (all_shots_made["x_coord"] <= -66.5)
    & (all_shots_made["y_coord"] >= -7)
    & (all_shots_made["y_coord"] <= 7)
)

# Medium Danger Zone (MD) - Right side
md_condition_right = (
    (
        (all_shots_made["x_coord"] >= 69)
        & (all_shots_made["x_coord"] <= 89)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= -7)
    )
    | (
        (all_shots_made["x_coord"] >= 54)
        & (all_shots_made["x_coord"] <= 69)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= 22)
    )
    | (
        (all_shots_made["x_coord"] >= 40)
        & (all_shots_made["x_coord"] <= 54)
        & (all_shots_made["y_coord"] >= -7)
        & (all_shots_made["y_coord"] <= 7)
    )
)

# Medium Danger Zone (MD) - Left side (mirrored)
md_condition_left = (
    (
        (all_shots_made["x_coord"] >= -89)
        & (all_shots_made["x_coord"] <= -69)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= -7)
    )
    | (
        (all_shots_made["x_coord"] >= -69)
        & (all_shots_made["x_coord"] <= -54)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= 22)
    )
    | (
        (all_shots_made["x_coord"] >= -54)
        & (all_shots_made["x_coord"] <= -40)
        & (all_shots_made["y_coord"] >= -7)
        & (all_shots_made["y_coord"] <= 7)
    )
)

# Low Danger Zone (LD) - Every other coordinate
ld_condition = ~(
    hd_condition_right | hd_condition_left | md_condition_right | md_condition_left
)

# Apply conditions to classify shots
conditions = [
    hd_condition_right | hd_condition_left,
    md_condition_right | md_condition_left,
    ld_condition,
]
values = ["HD", "MD", "LD"]

all_shots_made["shot_danger"] = np.select(conditions, values, default="LD")

# Create new columns for HD, MD, and LD shots
all_shots_made["hd_shot"] = (all_shots_made["shot_danger"] == "HD").astype(int)
all_shots_made["md_shot"] = (all_shots_made["shot_danger"] == "MD").astype(int)
all_shots_made["ld_shot"] = (all_shots_made["shot_danger"] == "LD").astype(int)


In [320]:
all_shots_made["game_time"] = (
    all_shots_made["period_number"] - 1
) * 1200 + all_shots_made["game_in_seconds"]

# Sort by game_id and game_time
# all_shots_made = all_shots_made.sort_values(by=['game_id', 'game_time'])

# Calculate time difference from the last shot in the same game
all_shots_made["time_since_last_shot"] = all_shots_made.groupby("game_id")[
    "game_time"
].diff()

# Define a rebound as a shot that follows another shot within 3 seconds
all_shots_made["is_rebound"] = (all_shots_made["time_since_last_shot"] <= 3).astype(int)

all_shots_made.head()


,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound
0,20132014,2013020001,53,01:14:00,18:46:00,507,missed-shot,13,1,10,35.0,-28.0,O,<NA>,<NA>,wrist,8471679,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,74,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,74,NaN,0
1,20132014,2013020001,54,01:36:00,18:24:00,506,shot-on-goal,18,1,8,-82.0,-22.0,O,<NA>,<NA>,wrist,8473503,0,1,<NA>,None,<NA>,<NA>,5 on 5,in net,96,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,96,22.0,0
2,20132014,2013020001,55,01:45:00,18:15:00,508,blocked-shot,21,1,10,57.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8475797,<NA>,5 on 5,in net,105,8,10,20132014,2013020001,100.0,O,hit,8,False,5.0,1.0,0,LD,0,0,1,105,9.0,0
3,20132014,2013020001,57,01:52:00,18:08:00,508,blocked-shot,26,1,10,37.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8468635,<NA>,5 on 5,in net,112,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0,LD,0,0,1,112,7.0,0
4,20132014,2013020001,59,02:24:00,17:36:00,506,shot-on-goal,36,1,8,-44.0,-5.0,O,<NA>,<NA>,wrist,8473503,0,2,<NA>,None,<NA>,<NA>,5 on 5,in net,144,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,144,32.0,0


### Scoring Chances

In [321]:

# Define conditions
conditions = [
    all_shots_made['hd_shot'] == 1,  # High Danger Shot
    all_shots_made['md_shot'] == 1,  # Medium Danger Shot
    all_shots_made['ld_shot'] == 1   # Low Danger Shot
]

# Define corresponding values
values = [3, 2, 1]  # Base values for scoring chances

# Apply np.select() to determine base scoring chance
all_shots_made['scoring_chance'] = np.select(conditions, values, default=1)

# Apply additional logic: If rush shot or rebound, increase scoring chance by 1
all_shots_made['scoring_chance'] += (all_shots_made['rush_shot'] == 1) | (all_shots_made['is_rebound'] == 1)
all_shots_made['scoring_chance'] -= all_shots_made['typeCode'] == 508

all_shots_made['scoring_chance'] = np.where(all_shots_made['scoring_chance'] >=2, 1, 0)
# Display result
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound,scoring_chance
0,20132014,2013020001,53,01:14:00,18:46:00,507,missed-shot,13,1,10,35.0,-28.0,O,<NA>,<NA>,wrist,8471679,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,74,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,74,NaN,0,0
1,20132014,2013020001,54,01:36:00,18:24:00,506,shot-on-goal,18,1,8,-82.0,-22.0,O,<NA>,<NA>,wrist,8473503,0,1,<NA>,None,<NA>,<NA>,5 on 5,in net,96,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,96,22.0,0,1
2,20132014,2013020001,55,01:45:00,18:15:00,508,blocked-shot,21,1,10,57.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8475797,<NA>,5 on 5,in net,105,8,10,20132014,2013020001,100.0,O,hit,8,False,5.0,1.0,0,LD,0,0,1,105,9.0,0,0
3,20132014,2013020001,57,01:52:00,18:08:00,508,blocked-shot,26,1,10,37.0,-31.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8468635,<NA>,5 on 5,in net,112,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0,LD,0,0,1,112,7.0,0,0
4,20132014,2013020001,59,02:24:00,17:36:00,506,shot-on-goal,36,1,8,-44.0,-5.0,O,<NA>,<NA>,wrist,8473503,0,2,<NA>,None,<NA>,<NA>,5 on 5,in net,144,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,144,32.0,0,1


In [322]:
scoring_chance_stats = all_shots_made[all_shots_made['scoring_chance']==1]
scoring_chance_stats.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound,scoring_chance
1,20132014,2013020001,54,01:36:00,18:24:00,506,shot-on-goal,18,1,8,-82.0,-22.0,O,<NA>,<NA>,wrist,8473503,0,1,<NA>,None,<NA>,<NA>,5 on 5,in net,96,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,96,22.0,0,1
4,20132014,2013020001,59,02:24:00,17:36:00,506,shot-on-goal,36,1,8,-44.0,-5.0,O,<NA>,<NA>,wrist,8473503,0,2,<NA>,None,<NA>,<NA>,5 on 5,in net,144,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,144,32.0,0,1
6,20132014,2013020001,63,02:58:00,17:02:00,506,shot-on-goal,45,1,10,55.0,-5.0,O,<NA>,<NA>,backhand,8471679,1,3,<NA>,None,<NA>,<NA>,5 on 5,in net,178,8,10,20132014,2013020001,176.0,D,hit,8,False,2.0,1.0,1,MD,0,1,0,178,30.0,0,1
10,20132014,2013020001,71,03:45:00,16:15:00,506,shot-on-goal,64,1,8,-85.0,18.0,O,<NA>,<NA>,wrist,8473503,3,4,<NA>,None,<NA>,<NA>,5 on 5,in net,225,8,10,20132014,2013020001,224.0,D,giveaway,10,False,1.0,1.0,1,LD,0,0,1,225,17.0,0,1
11,20132014,2013020001,72,03:55:00,16:05:00,506,shot-on-goal,66,1,10,81.0,-11.0,O,<NA>,<NA>,wrist,8471679,4,4,<NA>,None,<NA>,<NA>,5 on 5,in net,235,8,10,20132014,2013020001,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,235,10.0,0,1


In [323]:
scoring_chance_stats = scoring_chance_stats[['game_id', 'season_id','eventId', 'timeInPeriod', 'timeRemaining',  'team_id','typeCode', 'typeDescKey','shot_type', 'goalie',
        'situation', 'goalie_situation',
       'game_in_seconds', 'home_team', 'away_team', 'away_sog', 'home_sog','rush_shot', 'corsi',
       'hd_shot', 'md_shot', 'ld_shot', 'scoring_chance']]
scoring_chance_stats.head()

,game_id,season_id,eventId,timeInPeriod,timeRemaining,team_id,typeCode,typeDescKey,shot_type,goalie,situation,goalie_situation,game_in_seconds,home_team,away_team,away_sog,home_sog,rush_shot,corsi,hd_shot,md_shot,ld_shot,scoring_chance
1,2013020001,20132014,54,01:36:00,18:24:00,8,506,shot-on-goal,wrist,8473503,5 on 5,in net,96,8,10,0,1,0.0,1,0,1,0,1
4,2013020001,20132014,59,02:24:00,17:36:00,8,506,shot-on-goal,wrist,8473503,5 on 5,in net,144,8,10,0,2,0.0,1,0,1,0,1
6,2013020001,20132014,63,02:58:00,17:02:00,10,506,shot-on-goal,backhand,8471679,5 on 5,in net,178,8,10,1,3,1.0,1,0,1,0,1
10,2013020001,20132014,71,03:45:00,16:15:00,8,506,shot-on-goal,wrist,8473503,5 on 5,in net,225,8,10,3,4,1.0,1,0,0,1,1
11,2013020001,20132014,72,03:55:00,16:05:00,10,506,shot-on-goal,wrist,8471679,5 on 5,in net,235,8,10,4,4,0.0,1,0,1,0,1


In [324]:

# Default shooting team = team_id
scoring_chance_stats['shooting_team_id'] = scoring_chance_stats['team_id']

# Identify blocked shots (typeCode = 508)
mask = scoring_chance_stats['typeCode'] == 508

# For blocked shots, shooter is the OTHER team
home_block = (scoring_chance_stats['team_id'] == scoring_chance_stats['home_team'])
away_block = (scoring_chance_stats['team_id'] == scoring_chance_stats['away_team'])

# Shooter when home player blocked → shooter = away_team
scoring_chance_stats.loc[mask, 'shooting_team_id'] = scoring_chance_stats['away_team'].where(
    home_block, scoring_chance_stats['shooting_team_id']
)

# Shooter when away player blocked → shooter = home_team
scoring_chance_stats.loc[mask, 'shooting_team_id'] = scoring_chance_stats['home_team'].where(
    away_block, scoring_chance_stats['shooting_team_id']
)


scoring_chance_stats['scoring_chance_shot'] = np.where(
    scoring_chance_stats['typeCode'].isin([505, 506]),
    1, 
    0
)

scoring_chance_stats['scoring_chance_goal'] = np.where(
    scoring_chance_stats['typeCode'] == 505, 
    1, 
    0
)


scoring_chance_stats.head()

,game_id,season_id,eventId,timeInPeriod,timeRemaining,team_id,typeCode,typeDescKey,shot_type,goalie,situation,goalie_situation,game_in_seconds,home_team,away_team,away_sog,home_sog,rush_shot,corsi,hd_shot,md_shot,ld_shot,scoring_chance,shooting_team_id,scoring_chance_shot,scoring_chance_goal
1,2013020001,20132014,54,01:36:00,18:24:00,8,506,shot-on-goal,wrist,8473503,5 on 5,in net,96,8,10,0,1,0.0,1,0,1,0,1,8,1,0
4,2013020001,20132014,59,02:24:00,17:36:00,8,506,shot-on-goal,wrist,8473503,5 on 5,in net,144,8,10,0,2,0.0,1,0,1,0,1,8,1,0
6,2013020001,20132014,63,02:58:00,17:02:00,10,506,shot-on-goal,backhand,8471679,5 on 5,in net,178,8,10,1,3,1.0,1,0,1,0,1,10,1,0
10,2013020001,20132014,71,03:45:00,16:15:00,8,506,shot-on-goal,wrist,8473503,5 on 5,in net,225,8,10,3,4,1.0,1,0,0,1,1,8,1,0
11,2013020001,20132014,72,03:55:00,16:05:00,10,506,shot-on-goal,wrist,8471679,5 on 5,in net,235,8,10,4,4,0.0,1,0,1,0,1,10,1,0


In [325]:

# ==============================================
# Scoring Chances For (SCF) by Season + Team multi season
# ==============================================

# Make sure season_id is included
scf_stats = scoring_chance_stats[['season_id', 'shooting_team_id',
                                  'scoring_chance', 'scoring_chance_shot',
                                  'scoring_chance_goal']].copy()

# Group by season and team
scf_stats = (
    scf_stats
        .groupby(['season_id', 'shooting_team_id'], as_index=False)
        .agg(
            SCF=('scoring_chance', 'sum'),
            SCSF=('scoring_chance_shot', 'sum'),
            SCGF=('scoring_chance_goal', 'sum'),
        )
        .rename(columns={'shooting_team_id': 'team_id'})
)

# Preview
scf_stats.head()


,season_id,team_id,SCF,SCSF,SCGF
0,20132014,1,1968,1251,161
1,20132014,2,2482,1587,197
2,20132014,3,2423,1544,178
3,20132014,4,2180,1305,200
4,20132014,5,2084,1314,208


In [326]:


# ============================================================
# Scoring Chances Against (SCA) by Season + Team
# ============================================================

# --- Home SCA ---
sca_home = (
    scoring_chance_stats
        .groupby(['season_id', 'home_team'], as_index=False)
        .agg(SCA=('scoring_chance', 'sum'))
        .rename(columns={'home_team': 'team_id'})
)

# --- Away SCA ---
sca_away = (
    scoring_chance_stats
        .groupby(['season_id', 'away_team'], as_index=False)
        .agg(SCA=('scoring_chance', 'sum'))
        .rename(columns={'away_team': 'team_id'})
)

# --- Combine home + away ---
sca_stats = (
    pd.concat([sca_home, sca_away], ignore_index=True)
      .groupby(['season_id', 'team_id'], as_index=False)
      .agg(SCA=('SCA', 'sum'))
)

# --- Merge SCF + SCA ---
final_scoring_stats = (
    scf_stats.merge(sca_stats, on=['season_id', 'team_id'], how='left')
)

final_scoring_stats.head()


,season_id,team_id,SCF,SCSF,SCGF,SCA
0,20132014,1,1968,1251,161,3828.0
1,20132014,2,2482,1587,197,4969.0
2,20132014,3,2423,1544,178,4650.0
3,20132014,4,2180,1305,200,4353.0
4,20132014,5,2084,1314,208,4133.0


### Strength of Schedule

In [327]:

# ===============================================================
# STEP 1 — Compute Win% per season_id + team
# ===============================================================

team_stats = []

def calculate_team_win_percentage(season_df, team_abbrev):
    # Wins (home or away)
    wins = (
        len(season_df[(season_df["awayTeam.abbrev"] == team_abbrev) &
                      (season_df["awayTeam.score"] > season_df["homeTeam.score"])])
        +
        len(season_df[(season_df["homeTeam.abbrev"] == team_abbrev) &
                      (season_df["homeTeam.score"] > season_df["awayTeam.score"])])
    )

    # Total games
    games = len(season_df[(season_df["awayTeam.abbrev"] == team_abbrev) |
                          (season_df["homeTeam.abbrev"] == team_abbrev)])

    return wins / games if games > 0 else 0


# Loop by season_id
for season_id, season_df in history_summary.groupby("season_id"):
    team_abbrevs = pd.concat([
        season_df["awayTeam.abbrev"],
        season_df["homeTeam.abbrev"]
    ]).unique()

    for team_abbrev in team_abbrevs:
        win_pct = calculate_team_win_percentage(season_df, team_abbrev)

        team_stats.append({
            "season_id": season_id,
            "tri_code": team_abbrev,
            "win_percentage": win_pct
        })

team_stats_df = pd.DataFrame(team_stats)

# ===============================================================
# STEP 2 — Map Win% back into the main games dataframe
# ===============================================================

history_summary = history_summary.merge(
    team_stats_df.rename(columns={"tri_code": "awayTeam.abbrev"}),
    on=["season_id", "awayTeam.abbrev"],
    how="left"
).rename(columns={"win_percentage": "awayTeam_win_percentage"})

history_summary = history_summary.merge(
    team_stats_df.rename(columns={"tri_code": "homeTeam.abbrev"}),
    on=["season_id", "homeTeam.abbrev"],
    how="left"
).rename(columns={"win_percentage": "homeTeam_win_percentage"})


# ===============================================================
# STEP 3 — Calculate SOS per season_id + team_id
# ===============================================================

# Away Team SOS = average win% of opponents (home teams)
away_team_sos = (
    history_summary.groupby(["season_id", "awayTeam.abbrev"])["homeTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"awayTeam.abbrev": "tri_code",
                     "homeTeam_win_percentage": "SOS"})
)

# Home Team SOS = average win% of opponents (away teams)
home_team_sos = (
    history_summary.groupby(["season_id", "homeTeam.abbrev"])["awayTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"homeTeam.abbrev": "tri_code",
                     "awayTeam_win_percentage": "SOS"})
)

# Combine home + away appearances
team_sos = pd.concat([away_team_sos, home_team_sos], ignore_index=True)

# Final SOS per season + team
team_sos = (
    team_sos.groupby(["season_id", "tri_code"], as_index=False)["SOS"]
    .mean()
)

# Final output
team_sos.head()


,season_id,tri_code,SOS
0,20132014,ANA,0.494795
1,20132014,BOS,0.481559
2,20132014,BUF,0.507436
3,20132014,CAR,0.498959
4,20132014,CBJ,0.495687


In [328]:
team_sos = team_sos.merge(teams, on = 'tri_code', how = 'left')
strength = team_sos.copy()
strength.head()

,season_id,tri_code,SOS,team_id
0,20132014,ANA,0.494795,24.0
1,20132014,BOS,0.481559,6.0
2,20132014,BUF,0.507436,7.0
3,20132014,CAR,0.498959,12.0
4,20132014,CBJ,0.495687,29.0


In [329]:

strength = strength.dropna(subset=["team_id"])

strength['team_id'] = strength['team_id'].astype(int)

strength.head()

,season_id,tri_code,SOS,team_id
0,20132014,ANA,0.494795,24
1,20132014,BOS,0.481559,6
2,20132014,BUF,0.507436,7
3,20132014,CAR,0.498959,12
4,20132014,CBJ,0.495687,29


In [330]:
# full_schedule = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\season_schedule.csv"
# full_schedule_df = pd.read_csv(full_schedule, low_memory=False)
# full_schedule_df.tail()



In [331]:
# # Step 1: Filter out remaining games (future games only)
# future_games = full_schedule_df[full_schedule_df['gameState']=="FUT"]

# # Step 2: Map each opponent's win percentage
# future_games["awayTeam_win_percentage"] = future_games["awayTeam.abbrev"].map(
#     team_stats_df.set_index("team_abbrev")["win_percentage"]
# )
# future_games["homeTeam_win_percentage"] = future_games["homeTeam.abbrev"].map(
#     team_stats_df.set_index("team_abbrev")["win_percentage"]
# )

# # Step 3: Calculate SOS for remaining games
# # Calculate SOS for away teams
# away_team_remaining_sos = (
#     future_games.groupby(["season", "awayTeam.abbrev"])["homeTeam_win_percentage"]
#     .mean()
#     .reset_index()
# )
# away_team_remaining_sos.rename(columns={"homeTeam_win_percentage": "Remaining_SOS"}, inplace=True)

# # Calculate SOS for home teams
# home_team_remaining_sos = (
#     future_games.groupby(["season", "homeTeam.abbrev"])["awayTeam_win_percentage"]
#     .mean()
#     .reset_index()
# )
# home_team_remaining_sos.rename(columns={"awayTeam_win_percentage": "Remaining_SOS"}, inplace=True)

# # Step 4: Merge the Remaining SOS data for both away and home teams
# remaining_team_sos = pd.concat([away_team_remaining_sos, home_team_remaining_sos], axis=0)

# # Step 5: Compute the final remaining SOS per team
# remaining_team_sos = (
#     remaining_team_sos.groupby(["season", "awayTeam.abbrev"])["Remaining_SOS"]
#     .mean()
#     .reset_index()
# )

# # Rename columns for clarity
# remaining_team_sos.rename(
#     columns={"awayTeam.abbrev": "tri_code", "season": "season_id"}, inplace=True
# )

# remaining_team_sos.head()


In [332]:

# strength = team_sos.merge(remaining_team_sos, on='tri_code', how = 'inner')
# strength = strength[['tri_code', 'SOS', 'Remaining_SOS']]
# strength.head()




### Special Teams


In [333]:
# all_pk = all_pk[all_pk['season_id']!=20252026]
# all_pp = all_pp[all_pp['season_id']!=20252026]


In [334]:
all_pp.columns

Index(['season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses', 'pointPct',
       'points', 'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
       'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame', 'ppOpportunities',
       'ppOpportunitiesPerGame', 'ppTimeOnIcePerGame', 'shGoalsAgainst',
       'shGoalsAgainstPerGame', 'teamFullName', 'ties', 'wins'],
      dtype='object')

In [335]:
pp_num_cols = [ 'team_id', 'season_id',
    'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
    'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame',
    'ppOpportunities', 'ppOpportunitiesPerGame',
    'ppTimeOnIcePerGame', 'shGoalsAgainst',
    'shGoalsAgainstPerGame'
]

pk_num_cols =['team_id', 'season_id',
    'penaltyKillNetPct', 'penaltyKillPct',
       'pkNetGoals', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points',
       'pointsPct', 'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame', 'teamFullName', 'ties',
       'timesShorthanded', 'timesShorthandedPerGame', 'wins']

all_pp[pp_num_cols] = all_pp[pp_num_cols].fillna(0)
all_pk[pk_num_cols] = all_pk[pk_num_cols].fillna(0)

C:\Users\kjchr\AppData\Local\Temp\ipykernel_26016\1363811887.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_pk[pk_num_cols] = all_pk[pk_num_cols].fillna(0)


In [336]:
pp_stats = all_pp.copy()

pp_stats = pp_stats[['game_id', 'season_id', 'team_id', 'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
       'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame', 'ppOpportunities',
       'ppOpportunitiesPerGame', 'ppTimeOnIcePerGame', 'shGoalsAgainst',
       'shGoalsAgainstPerGame', 'opponentTeamAbbrev']]

pp_stats.head()

,game_id,season_id,team_id,powerPlayGoalsFor,powerPlayNetPct,powerPlayPct,ppGoalsPerGame,ppNetGoals,ppNetGoalsPerGame,ppOpportunities,ppOpportunitiesPerGame,ppTimeOnIcePerGame,shGoalsAgainst,shGoalsAgainstPerGame,opponentTeamAbbrev
0,2013020001,20132014,10,1,0.142857,0.14285,1.0,1,1.0,7,7.0,700.0,0,0.0,MTL
1,2013020001,20132014,8,0,-0.250000,0.00000,0.0,-1,-1.0,4,4.0,405.0,1,1.0,TOR
2,2013020002,20132014,15,3,0.500000,0.50000,3.0,3,3.0,6,6.0,388.0,0,0.0,CHI
3,2013020002,20132014,16,1,0.250000,0.25000,1.0,1,1.0,4,4.0,387.0,0,0.0,WSH
4,2013020003,20132014,22,2,0.400000,0.40000,2.0,2,2.0,5,5.0,399.0,0,0.0,WPG


In [337]:
# ==============================================================
# AGGREGATE PP STATS BY TEAM + SEASON
# ==============================================================

pp_stats = pp_stats[['game_id', 'season_id', 'team_id', 
                     'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
                     'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame',
                     'ppOpportunities', 'ppOpportunitiesPerGame',
                     'ppTimeOnIcePerGame', 'shGoalsAgainst',
                     'shGoalsAgainstPerGame', 'opponentTeamAbbrev']].copy()

# --------------------------------------------------------------
# 1. Aggregate raw totals (correct way)
# --------------------------------------------------------------
agg_pp = (
    pp_stats.groupby(['season_id', 'team_id'], as_index=False)
    .agg(
        games_played=('game_id', 'nunique'),
        powerPlayGoalsFor=('powerPlayGoalsFor', 'sum'),
        shGoalsAgainst=('shGoalsAgainst', 'sum'),
        ppOpportunities=('ppOpportunities', 'sum'),
        ppNetGoals=('ppNetGoals', 'sum'),
        ppTimeOnIce=('ppTimeOnIcePerGame', 'sum'),
    )
)

# --------------------------------------------------------------
# 2. Recompute correct percentages 
# --------------------------------------------------------------

# PP% = PPGF / PPO
agg_pp['powerPlayPct'] = (
    agg_pp['powerPlayGoalsFor'] / agg_pp['ppOpportunities']
).replace([np.inf, -np.inf], np.nan).round(4) * 100

# Net PP% = (PPGF - SHGA) / PPO
agg_pp['powerPlayNetPct'] = (
    (agg_pp['powerPlayGoalsFor'] - agg_pp['shGoalsAgainst'])
    / agg_pp['ppOpportunities']
).replace([np.inf, -np.inf], np.nan).round(4) * 100

# --------------------------------------------------------------
# 3. Recompute per-game metrics
# --------------------------------------------------------------

agg_pp['ppGoalsPerGame'] = (
    agg_pp['powerPlayGoalsFor'] / agg_pp['games_played']
).round(3)

agg_pp['ppOpportunitiesPerGame'] = (
    agg_pp['ppOpportunities'] / agg_pp['games_played']
).round(3)

agg_pp['shGoalsAgainstPerGame'] = (
    agg_pp['shGoalsAgainst'] / agg_pp['games_played']
).round(3)

agg_pp['ppNetGoalsPerGame'] = (
    agg_pp['ppNetGoals'] / agg_pp['games_played']
).round(3)

agg_pp['ppTimeOnIcePerGame'] = (
    agg_pp['ppTimeOnIce'] / agg_pp['games_played']
).round(3)

# Clean final DataFrame
numeric_cols = agg_pp.select_dtypes(include=['float']).columns
agg_pp[numeric_cols] = agg_pp[numeric_cols].round(3)

# Final dataset
agg_pp.head()


,season_id,team_id,games_played,powerPlayGoalsFor,shGoalsAgainst,ppOpportunities,ppNetGoals,ppTimeOnIce,powerPlayPct,powerPlayNetPct,ppGoalsPerGame,ppOpportunitiesPerGame,shGoalsAgainstPerGame,ppNetGoalsPerGame,ppTimeOnIcePerGame
0,20132014,1,82,47,4,241,43,24595.0,19.50,17.84,0.573,2.939,0.049,0.524,299.939
1,20132014,2,82,49,7,276,42,27408.0,17.75,15.22,0.598,3.366,0.085,0.512,334.244
2,20132014,3,82,48,7,264,41,26313.0,18.18,15.53,0.585,3.220,0.085,0.500,320.890
3,20132014,4,82,58,11,294,47,28986.0,19.73,15.99,0.707,3.585,0.134,0.573,353.488
4,20132014,5,82,65,6,278,59,26652.0,23.38,21.22,0.793,3.390,0.073,0.720,325.024


In [338]:
pk_stats = all_pk.copy()

pk_stats = pk_stats[['game_id', 'season_id', 'team_id',  'penaltyKillNetPct', 'penaltyKillPct',
       'pkNetGoals', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points',
       'pointsPct', 'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame','timesShorthanded', 'timesShorthandedPerGame', 'opponentTeamAbbrev']]

pk_stats.head()

,game_id,season_id,team_id,penaltyKillNetPct,penaltyKillPct,pkNetGoals,pkNetGoalsPerGame,pkTimeOnIcePerGame,points,pointsPct,ppGoalsAgainst,ppGoalsAgainstPerGame,shGoalsFor,shGoalsForPerGame,timesShorthanded,timesShorthandedPerGame,opponentTeamAbbrev
0,2013020001,20132014,10,1.250000,1.00000,1,1.0,405.0,2,1.0,0,0.0,1,1.0,4,4.0,MTL
1,2013020001,20132014,8,0.857142,0.85714,-1,-1.0,700.0,0,0.0,1,1.0,0,0.0,7,7.0,TOR
2,2013020002,20132014,16,0.500000,0.50000,-3,-3.0,388.0,2,1.0,3,3.0,0,0.0,6,6.0,WSH
3,2013020002,20132014,15,0.750000,0.75000,-1,-1.0,387.0,0,0.0,1,1.0,0,0.0,4,4.0,CHI
4,2013020003,20132014,22,0.750000,0.75000,-1,-1.0,400.0,0,0.0,1,1.0,0,0.0,4,4.0,WPG


In [339]:
# ==============================================
# AGGREGATE PK STATS BY TEAM + SEASON
# ==============================================

# Start with your selected PK stats
pk_stats = pk_stats[['game_id', 'season_id', 'team_id', 'penaltyKillNetPct', 'penaltyKillPct',
       'pkNetGoals', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points',
       'pointsPct', 'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame','timesShorthanded', 'timesShorthandedPerGame', 'opponentTeamAbbrev']].copy()

# Group by team + season
agg_pk = (
    pk_stats.groupby(['season_id', 'team_id'], as_index=False)
    .agg(
        games_played=('game_id', 'nunique'),
        ppGoalsAgainst=('ppGoalsAgainst', 'sum'),
        shGoalsFor=('shGoalsFor', 'sum'),
        timesShorthanded=('timesShorthanded', 'sum'),
        pkNetGoals=('pkNetGoals', 'sum'),
        pkTimeOnIce=('pkTimeOnIcePerGame', 'sum'),   # total minutes short-handed
        points=('points', 'sum'),
    )
)

# ----------------------------------------------
# RE-CALCULATE ALL % METRICS PROPERLY
# ----------------------------------------------

# Classic PK% = 1 - PPGA / TSH
agg_pk['penaltyKillPct'] = (
    1 - (agg_pk['ppGoalsAgainst'] / agg_pk['timesShorthanded'])
).round(4) * 100

# Net PK% = (TSH - PPGA + SHGF) / TSH
agg_pk['penaltyKillNetPct'] = (
    (agg_pk['timesShorthanded'] - agg_pk['ppGoalsAgainst'] + agg_pk['shGoalsFor'])
    / agg_pk['timesShorthanded']
).round(4) * 100

# Per-game metrics (recomputed)
agg_pk['ppGoalsAgainstPerGame'] = agg_pk['ppGoalsAgainst'] / agg_pk['games_played']
agg_pk['shGoalsForPerGame'] = agg_pk['shGoalsFor'] / agg_pk['games_played']
agg_pk['timesShorthandedPerGame'] = agg_pk['timesShorthanded'] / agg_pk['games_played']
agg_pk['pkNetGoalsPerGame'] = agg_pk['pkNetGoals'] / agg_pk['games_played']
agg_pk['pkTimeOnIcePerGame'] = agg_pk['pkTimeOnIce'] / agg_pk['games_played']

# Points%
agg_pk['pointsPct'] = agg_pk['points'] / (agg_pk['games_played'] * 2)

# Round final numbers
numeric_cols = agg_pk.select_dtypes(include=['float']).columns
agg_pk[numeric_cols] = agg_pk[numeric_cols].round(3)

# Final dataset
agg_pk.head()


,season_id,team_id,games_played,ppGoalsAgainst,shGoalsFor,timesShorthanded,pkNetGoals,pkTimeOnIce,points,penaltyKillPct,penaltyKillNetPct,ppGoalsAgainstPerGame,shGoalsForPerGame,timesShorthandedPerGame,pkNetGoalsPerGame,pkTimeOnIcePerGame,pointsPct
0,20132014,1,82,36,9,264,-27,27808.0,88,86.36,89.77,0.439,0.110,3.220,-0.329,339.122,0.537
1,20132014,2,82,58,8,265,-50,26053.0,79,78.11,81.13,0.707,0.098,3.232,-0.610,317.720,0.482
2,20132014,3,82,34,10,232,-24,23152.0,96,85.34,89.66,0.415,0.122,2.829,-0.293,282.341,0.585
3,20132014,4,82,48,8,316,-40,32536.0,94,84.81,87.34,0.585,0.098,3.854,-0.488,396.780,0.573
4,20132014,5,82,38,4,254,-34,25535.0,109,85.04,86.61,0.463,0.049,3.098,-0.415,311.402,0.665


In [340]:
special_teams = agg_pp.merge(
    agg_pk,
    on=['season_id', 'team_id'],
    how='inner',
    suffixes=('_pp', '_pk')   # optional — only needed if any column names overlap
)


special_teams=special_teams.drop(columns=['points'])
special_teams.head()

,season_id,team_id,games_played_pp,powerPlayGoalsFor,shGoalsAgainst,ppOpportunities,ppNetGoals,ppTimeOnIce,powerPlayPct,powerPlayNetPct,ppGoalsPerGame,ppOpportunitiesPerGame,shGoalsAgainstPerGame,ppNetGoalsPerGame,ppTimeOnIcePerGame,games_played_pk,ppGoalsAgainst,shGoalsFor,timesShorthanded,pkNetGoals,pkTimeOnIce,penaltyKillPct,penaltyKillNetPct,ppGoalsAgainstPerGame,shGoalsForPerGame,timesShorthandedPerGame,pkNetGoalsPerGame,pkTimeOnIcePerGame,pointsPct
0,20132014,1,82,47,4,241,43,24595.0,19.50,17.84,0.573,2.939,0.049,0.524,299.939,82,36,9,264,-27,27808.0,86.36,89.77,0.439,0.110,3.220,-0.329,339.122,0.537
1,20132014,2,82,49,7,276,42,27408.0,17.75,15.22,0.598,3.366,0.085,0.512,334.244,82,58,8,265,-50,26053.0,78.11,81.13,0.707,0.098,3.232,-0.610,317.720,0.482
2,20132014,3,82,48,7,264,41,26313.0,18.18,15.53,0.585,3.220,0.085,0.500,320.890,82,34,10,232,-24,23152.0,85.34,89.66,0.415,0.122,2.829,-0.293,282.341,0.585
3,20132014,4,82,58,11,294,47,28986.0,19.73,15.99,0.707,3.585,0.134,0.573,353.488,82,48,8,316,-40,32536.0,84.81,87.34,0.585,0.098,3.854,-0.488,396.780,0.573
4,20132014,5,82,65,6,278,59,26652.0,23.38,21.22,0.793,3.390,0.073,0.720,325.024,82,38,4,254,-34,25535.0,85.04,86.61,0.463,0.049,3.098,-0.415,311.402,0.665


In [341]:

special_teams.columns

Index(['season_id', 'team_id', 'games_played_pp', 'powerPlayGoalsFor',
       'shGoalsAgainst', 'ppOpportunities', 'ppNetGoals', 'ppTimeOnIce',
       'powerPlayPct', 'powerPlayNetPct', 'ppGoalsPerGame',
       'ppOpportunitiesPerGame', 'shGoalsAgainstPerGame', 'ppNetGoalsPerGame',
       'ppTimeOnIcePerGame', 'games_played_pk', 'ppGoalsAgainst', 'shGoalsFor',
       'timesShorthanded', 'pkNetGoals', 'pkTimeOnIce', 'penaltyKillPct',
       'penaltyKillNetPct', 'ppGoalsAgainstPerGame', 'shGoalsForPerGame',
       'timesShorthandedPerGame', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame',
       'pointsPct'],
      dtype='object')

In [342]:
special_teams = special_teams.rename(
            columns={"teamId": "team_id",

                "powerPlayGoalsFor": "pp_gf",
                "powerPlayNetPct": "pp_net_pct",
                "powerPlayPct": "pp_pct",
                "ppGoalsPerGame": "ppg_g",
                "ppNetGoals": "ppg_net",
                "ppNetGoalsPerGame": "ppg_net_g",
                "ppOpportunities": "pp_opp",
                "ppOpportunitiesPerGame": "pp_opp_g",
                "penaltyKillNetPct": "pk_net_pct",
                "penaltyKillPct": "pk_pct",
                "pkNetGoals": "pkg_net",
                "pkNetGoalsPerGame": "pkg_net_g",
                "ppGoalsAgainst": "ppga",
                "ppGoalsAgainstPerGame": "ppga_g",
                "shGoalsFor": "shgf",
                "shGoalsForPerGame": "shgf_g",
            }
        )
special_teams.head()

,season_id,team_id,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct
0,20132014,1,82,47,4,241,43,24595.0,19.50,17.84,0.573,2.939,0.049,0.524,299.939,82,36,9,264,-27,27808.0,86.36,89.77,0.439,0.110,3.220,-0.329,339.122,0.537
1,20132014,2,82,49,7,276,42,27408.0,17.75,15.22,0.598,3.366,0.085,0.512,334.244,82,58,8,265,-50,26053.0,78.11,81.13,0.707,0.098,3.232,-0.610,317.720,0.482
2,20132014,3,82,48,7,264,41,26313.0,18.18,15.53,0.585,3.220,0.085,0.500,320.890,82,34,10,232,-24,23152.0,85.34,89.66,0.415,0.122,2.829,-0.293,282.341,0.585
3,20132014,4,82,58,11,294,47,28986.0,19.73,15.99,0.707,3.585,0.134,0.573,353.488,82,48,8,316,-40,32536.0,84.81,87.34,0.585,0.098,3.854,-0.488,396.780,0.573
4,20132014,5,82,65,6,278,59,26652.0,23.38,21.22,0.793,3.390,0.073,0.720,325.024,82,38,4,254,-34,25535.0,85.04,86.61,0.463,0.049,3.098,-0.415,311.402,0.665


### FINAL DF

In [343]:
final_standings = final_standings.merge(teams, on='tri_code', how = 'left')
final_standings.head()

,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,team_id
0,20132014,BOS,Boston Bruins,Eastern,1,82,117,0.658537,0.713415,51,84,261,177,41,31,41,23,7,L,1,0.756098,0.560976,3.182927,2.158537,117.0,6
1,20132014,ANA,Anaheim Ducks,Western,1,82,116,0.658537,0.707317,51,57,266,209,41,29,41,25,8,W,4,0.707317,0.609756,3.243902,2.548780,116.0,24
2,20132014,COL,Colorado Avalanche,Western,2,82,112,0.634146,0.682927,47,30,250,220,41,26,41,26,7,OT,1,0.634146,0.634146,3.048780,2.682927,112.0,21
3,20132014,STL,St. Louis Blues,Western,3,82,111,0.634146,0.676829,43,57,248,191,41,28,41,24,-13,L,6,0.682927,0.585366,3.024390,2.329268,111.0,19
4,20132014,SJS,San Jose Sharks,Western,4,82,111,0.621951,0.676829,41,49,249,200,41,29,41,22,1,W,2,0.707317,0.536585,3.036585,2.439024,111.0,28


In [344]:
team_goalie_stats=team_goalie_stats.merge(teams, on='tri_code', how = 'left')
team_goalie_stats.head()

,season_id,tri_code,team_savePct,team_GAA,team_id
0,20132014,ANA,0.915279,2.405251,24
1,20142015,ANA,0.909704,2.564931,24
2,20152016,ANA,0.918772,2.218657,24
3,20162017,ANA,0.920788,2.325711,24
4,20172018,ANA,0.926456,2.405461,24


In [345]:
history_final_df = (
    final_standings[
        ['season_id', 'tri_code', 'team_name', 'conferenceName',
       'conferenceSequence', 'gamesPlayed', 'points', 'winPctg', 'pointPctg',
       'regulationPlusOtWins', 'goalDifferential', 'goalFor', 'goalAgainst',
       'homeGamesPlayed', 'homeWins', 'roadGamesPlayed', 'roadWins',
       'l10GoalDifferential', 'streakCode', 'streakCount', 'home_win_pctg',
       'road_win_pctg', 'gf_g', 'ga_g', 'projected_points', 'team_id']
    ]
    .merge(clinch_history, on=['team_id', 'tri_code', 'season_id'], how='left')
    .merge(strength, on=['team_id','tri_code','season_id'], how='left')
    .merge(special_teams, on=['team_id','season_id'], how="left")
    .merge(rush_stats, on=['team_id','tri_code','season_id'], how='left')

    .merge(team_goalie_stats, on=['team_id','tri_code','season_id'], how='left')
    .merge(shooting_stats,on=['team_id','season_id'], how='left')
    .merge(final_scoring_stats, on=['team_id','season_id'], how='left')
)
today = date.today()
history_final_df['run_date'] = today
history_final_df.head()

,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,team_id,clinch_type,clinch_date,SOS,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct,rush_shots_for,rush_shots_against,rush_shot_differential,team_savePct,team_GAA,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo,SCF,SCSF,SCGF,SCA,run_date
0,20132014,BOS,Boston Bruins,Eastern,1,82,117,0.658537,0.713415,51,84,261,177,41,31,41,23,7,L,1,0.756098,0.560976,3.182927,2.158537,117.0,6,x,2014-03-21,0.481559,82.0,50.0,6.0,230.0,44.0,22133.0,21.74,19.13,0.610,2.805,0.073,0.537,269.915,82.0,43.0,9.0,263.0,-34.0,26355.0,83.65,87.07,0.524,0.110,3.207,-0.415,321.402,0.713,441.0,426.0,15.0,0.928687,2.057238,7.29,3636.0,3262.0,5.55,94.45,101.74,2205.0,1391.0,208.0,4260.0,2025-12-04
1,20132014,ANA,Anaheim Ducks,Western,1,82,116,0.658537,0.707317,51,57,266,209,41,29,41,25,8,W,4,0.707317,0.609756,3.243902,2.548780,116.0,24,x,2014-03-25,0.494795,82.0,44.0,8.0,275.0,36.0,28061.0,16.00,13.09,0.537,3.354,0.098,0.439,342.207,82.0,48.0,10.0,270.0,-38.0,26739.0,82.22,85.93,0.585,0.122,3.293,-0.463,326.085,0.707,461.0,434.0,27.0,0.915279,2.405251,7.58,3630.0,3479.0,6.27,93.73,101.31,2539.0,1530.0,229.0,4829.0,2025-12-04
2,20132014,COL,Colorado Avalanche,Western,2,82,112,0.634146,0.682927,47,30,250,220,41,26,41,26,7,OT,1,0.634146,0.634146,3.048780,2.682927,112.0,21,x,2014-03-29,0.501636,82.0,50.0,1.0,252.0,49.0,25502.0,19.84,19.44,0.610,3.073,0.012,0.598,311.000,82.0,48.0,3.0,249.0,-45.0,25545.0,80.72,81.93,0.585,0.037,3.037,-0.549,311.524,0.683,356.0,424.0,-68.0,0.921748,2.526457,7.76,3274.0,3764.0,5.92,94.08,101.84,2095.0,1320.0,203.0,4433.0,2025-12-04
3,20132014,STL,St. Louis Blues,Western,3,82,111,0.634146,0.676829,43,57,248,191,41,28,41,24,-13,L,6,0.682927,0.585366,3.024390,2.329268,111.0,19,x,2014-03-22,0.499703,82.0,56.0,6.0,283.0,50.0,27065.0,19.79,17.67,0.683,3.451,0.073,0.610,330.061,82.0,42.0,6.0,293.0,-36.0,29617.0,85.67,87.71,0.512,0.073,3.573,-0.439,361.183,0.677,424.0,384.0,40.0,0.915620,2.193268,7.49,3472.0,3077.0,6.50,93.50,100.99,2221.0,1328.0,212.0,4223.0,2025-12-04
4,20132014,SJS,San Jose Sharks,Western,4,82,111,0.621951,0.676829,41,49,249,200,41,29,41,22,1,W,2,0.707317,0.536585,3.036585,2.439024,111.0,28,x,2014-03-24,0.499108,82.0,50.0,6.0,291.0,44.0,28427.0,17.18,15.12,0.610,3.549,0.073,0.537,346.671,82.0,33.0,8.0,219.0,-25.0,21707.0,84.93,88.58,0.402,0.098,2.671,-0.305,264.720,0.677,529.0,458.0,71.0,0.917395,2.259612,6.43,4056.0,3265.0,6.40,93.60,100.03,2593.0,1664.0,217.0,4753.0,2025-12-04


In [346]:
#For the thanksgiving model, filter out the current season data before export.

history_final_df = history_final_df[history_final_df['season_id']!=20252026]


In [347]:

# Export
if history_final_df is not None:
    export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Thanksgiving Prep\final_agg_data.csv"
    os.makedirs(os.path.dirname(export_path), exist_ok=True)
    history_final_df.to_csv(export_path, index=False)
    print("✅ Final dataset exported successfully!")
else:
    print("❌ No data returned.")

✅ Final dataset exported successfully!
